# Teach More, Keep the Signature — Fresh32 + V5 candidate

**Historical confirmed best: 0.642, exact V3 (scriptVersionId362149717).** That result belongs to the preserved previous pipeline, not to this new candidate. The current candidate is unscored until its own official submission completes.

Derived from [Dhruv Tibarewal nlm-v27](https://www.kaggle.com/code/dhruvtibarewal/nlm-v27), [lightningv08](https://www.kaggle.com/code/lightningv08/0-642-lora-llama-qwen3-8-27b) and [Shaksham Karkiii sharper-blend / public stack](https://www.kaggle.com/code/shakshamkarkiii/ntl-public-stack). Kirill Tymchishin old4404 spectral readouts retain their original checkpoint and provenance.

Fixed **72.25% nlm-inspired +12.75% old4404 spectral +15% fresh Qwen3-32B** probability shares before one final balancing. Three V5 checkpoints per family (fold0,fold2,fullfit; nine total) are used. This intentional reduction differs from the eighteen-checkpoint nlm-v27. The 27B branch is replaced completely by an independent compatible32B adapter.

The fresh32B model trains one external epoch on OLD4404 plus all1677 new nonempty exact-identity responses, then two epochs on all2699 official training rows. No quality, length, duplicate or complete-group selection was applied to the new labelled data. Raw responses and raw datasets remain private. The fixed SVC was prepared on CPU. There is no claimed improvement or full-ensemble OOF result.

Only the exact known official three-row public fixture receives a format preview before torch/CUDA. Every other input performs genuine training/loading and inference. The whole pipeline has one8.5h target inside the competition9h limit; a timeout is a failure, not a substitute prediction.

**Built with Llama.** Llama3.1-8B Base is from the licensed NousResearch redistribution, pinned to1f47e50cdbe801ad8a5174156ec3a0655108fb9f. Selected file identities match the official Meta repository d04e592bb4f6aa9cfee91e2e20afa771667e1d4b. License, use policy and Meta attribution are retained. No Instruct substitution or27B adapter is used.


In [ ]:
import os,time
os.environ["MERGE26_STARTED_WALL"]=str(time.time())
"""Only the exact known public fixture may bypass the genuine pipeline."""
import csv
import hashlib
import json
from pathlib import Path

PUBLIC=[('00353fb1c541',321,'0c306cbd5d401399327e4cb46f766928'),
 ('005f5a9a6139',764,'c2853bf2b68bdd808a8a05cec2439b5d'),
 ('0062a53d5487',1077,'f8c583ac9cbc0806bee20f96c9124159')]

def matches(records):
    return [(r['id'],len(r['text'].encode('utf-8')),
        hashlib.blake2s(r['text'].encode('utf-8'),digest_size=16).hexdigest()) for r in records]==PUBLIC

def write_if_exact(data,output):
    with (Path(data)/'test.csv').open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=['id','text']:return False
        rows=[]
        for row in reader:
            rows.append(row)
            if len(rows)>3:return False
    if not matches(rows):return False
    import pandas as pd
    pd.DataFrame(dict(id=[r['id'] for r in rows],model=['claude-fable-5']*3)).to_parquet(output,index=False)
    Path(output).with_suffix('.mode.json').write_text(json.dumps(dict(
        mode='exact_public_format_only',GPU_model_loaded=False,hidden_runtime_proven=False)))
    return True

from pathlib import Path
DATA=Path('/kaggle/input/competitions/name-that-llm')
OUTPUT=Path('/kaggle/working/submission.parquet')
PREVIEW=write_if_exact(DATA,OUTPUT)
print('EXACT_PUBLIC_FORMAT_PREVIEW' if PREVIEW else 'REAL_TRAIN32_PLUS_ENSEMBLE')


In [ ]:
if not PREVIEW:
    OUTPUT.unlink(missing_ok=True)
    root=Path('/kaggle/working/merge26_runtime');root.mkdir(exist_ok=True)
    (root/'comp_stack.py').write_text('"""Complementary frozen-feature stack (public Apache-2.0 notebooks by prvsiyan / lucasboesen / vnton37nguyn / kirilltym,\nas embedded in the public 0.6xx ATLAS notebooks). Run as a separate process; writes its raw log-probability blend."""\n# ============================================================ 0. configuration\nimport gc, glob, json, os, pathlib, re, time\n\n_merge_original_glob=glob.glob\ndef _merge_scoped_glob(pattern,*args,**kwargs):\n    found=_merge_original_glob(pattern,*args,**kwargs)\n    if str(pattern).startswith(\'/kaggle/input\'):\n        found=[p for p in found if not p.startswith(\'/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets\')\n            and not p.startswith(\'/kaggle/input/datasets/kirilltym/ntl-qwen3-32b-nf4-pinned\')\n            and not p.startswith(\'/kaggle/input/datasets/kirilltym/ntl-llama31-8b-base-pinned\')\n            and not p.startswith(\'/kaggle/input/notebooks/kirilltym/ntl-free-proxy-data-pilot\')\n            and not p.startswith(\'/kaggle/input/datasets/kirilltym/ntl-private-unfiltered-2026-10-26\')]\n    return found\nglob.glob=_merge_scoped_glob\n\nimport numpy as np, pandas as pd\ntry:\n    import torch\nexcept ImportError:   # local smoke without torch\n    torch = None\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.sparse import csr_matrix, diags, hstack, vstack\nfrom scipy.spatial.distance import squareform\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import LinearSVC\n\nSMOKE = os.environ.get(\'NTL_SMOKE\') == \'1\'   # local code-path check: pseudo features instead of checkpoints, CPU\nT0 = time.time()\nLABELS = sorted([\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\n                 \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\n                 \'claude-fable-5\', \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\',\n                 \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\', \'kimi-k3\'])\nK = len(LABELS)\nLAB2I = {l: i for i, l in enumerate(LABELS)}\nFAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in (\'claude\', \'gpt\', \'gemini\', \'glm\')}\n\n# ---- test-pool grouping\nGROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18   # mixed content + GLM-L24 distance, average linkage\nSTYLE_GROUP_THRESH = 0.90                                           # stylegram\'s own median-normalised content distance\n\n# ---- blend weights: log-probability multipliers of the 0.563 notebook (its L-BFGS fit on prompt-grouped OOF)\nW = {\'stylegram\': 1.516, \'stylometry_lgb\': 0.063, \'ultrabert\': 0.095,\n     \'q3_8b\': 0.0, \'q25_7b\': 0.0, \'q3_4b\': 0.0, \'glm9b\': 0.0, \'debv3l\': 0.05, \'electl\': 0.032, \'desk\': 0.132,\n     \'lora24_public\': 0.177, \'lora24_seed7\': 0.113, \'lora24_claude\': 0.177, \'lora20_q25_instyle\': 0.17,\n     \'tokattn_q35\': 0.153, \'tokattn_deb\': 0.185}\nTWIN_W = {\'q3_8b\': 1.5 * 0.219, \'q25_7b\': 1.5 * 0.141, \'q3_4b\': 1.5 * 0.083, \'glm9b\': 1.5 * 0.110, \'debv3l\': 0.25}\nLORA_TWIN_W = {\'lora24_public\': 0.15, \'lora24_claude\': 0.15}\nGATE_LAM = {\'claude_style\': 1.5, \'claude_lora\': 1.0, \'family\': 0.25}\nBALANCE_ALPHA = 0.5\nEXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5\n\n# ---- the addition: shared-prompt Gaussian group heads on four measured views, weight = the centred twin of the same backbone,\n#      reduction / temperatures from our production fit (research/gaussian/production_weights_v8g.json, 135 training prompts)\nGAUSS_HEADS = {\'gauss_q3_8b_L22\': dict(src=\'q3_8b\', part=\'L22_mean\', d=384, T1=2.351, Tg=2.306, w=TWIN_W[\'q3_8b\']),\n               \'gauss_glm9b_L8\': dict(src=\'glm9b\', part=\'L8_mean\', d=384, T1=2.722, Tg=2.596, w=TWIN_W[\'glm9b\']),\n               \'gauss_q3_4b_L4\': dict(src=\'q3_4b\', part=\'L4_mean\', d=384, T1=2.725, Tg=2.578, w=TWIN_W[\'q3_4b\']),\n               \'gauss_debv3l_L24\': dict(src=\'debv3l\', part=\'L24_mean\', d=384, T1=3.112, Tg=3.069, w=TWIN_W[\'debv3l\'])}\nGAUSS_SETTING = dict(red=\'wpca\', se_shrink=\'lw\', sp_shrink=\'lw\', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)\n\n# ---- member specifications\nLM_HEADS = [  # key, model_type, hidden size, parts of the plain head [(layer, pool)], extra parts extracted in the same pass, C\n    (\'glm9b\', \'glm4\', 4096, [(8, \'mean\'), (3, \'mean\')], [(GROUP_LAYER, \'mean\')], 0.0025),   # first: its L24 builds the groups\n    (\'q3_8b\', \'qwen3\', 4096, [(7, \'mean\'), (22, \'mean\')], [], 0.0025),\n    (\'q25_7b\', \'qwen2\', 3584, [(6, \'mean\'), (20, \'mean\')], [], 0.0025),\n    (\'q3_4b\', \'qwen3\', 2560, [(4, \'style\'), (7, \'style\')], [(4, \'mean\')], 0.0025)]\nENC_HEADS = [  # key, input directory name, parts, C\n    (\'debv3l\', \'deberta-v3-large\', [(24, \'mean\'), (6, \'mean\')], 0.005),\n    (\'electl\', \'electra-large-discriminator\', [(21, \'mean\'), (24, \'style\')], 0.005),\n    (\'desk\', \'desklib-detector\', [(23, \'mean\'), (23, \'style\')], 0.005)]\nLORA_RUNS = [  # (trunk model_type, hidden, kept layers), pooled layers, [(key, checkpoint marker, norms file, anchor features file)]\n    ((\'qwen3\', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [\n        (\'lora24_public\', \'**/lora24/final/COMPLETE.json\', \'heads.safetensors\', \'anchor_features.npz\'),\n        (\'lora24_seed7\', \'**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json\', \'norms.safetensors\', \'valid_features.npz\'),\n        (\'lora24_claude\', \'**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json\', \'heads.safetensors\', \'anchor_features.npz\')]),\n    ((\'qwen2\', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [\n        (\'lora20_q25_instyle\', \'**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json\', \'norms.safetensors\', \'valid_features.npz\')])]\nLORA_MAX_LEN, LORA_BS = 447, 8\nTOKATTN_HEADS = [(\'tokattn_q35\', \'tokattn_final.pt\', \'qwen3_5\'), (\'tokattn_deb\', \'tokattn_debl23_final.pt\', \'deberta\')]\nif os.environ.get(\'NTL_DROP_Q35\') == \'1\':   # Qwen3.5-9B token-attention member: 1.7-2.7 h fixed cost on Kaggle, weight 0.153 of ~3.3\n    TOKATTN_HEADS = [t for t in TOKATTN_HEADS if t[0] != \'tokattn_q35\']\n\n# ---- data\nif SMOKE:\n    _all = pd.read_csv(os.environ[\'NTL_DATA\'] + \'/train.csv\')\n    _p = _all[\'prompt_id\'].drop_duplicates().sample(frac=1, random_state=0).tolist()\n    train = _all[_all[\'prompt_id\'].isin(_p[:30])].reset_index(drop=True)\n    test = _all[_all[\'prompt_id\'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)\n    test = pd.concat([test[test[\'prompt_id\'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)\n    if os.environ.get(\'NTL_TINY\') == \'1\':\n        test = test.head(3).reset_index(drop=True)\n    SMOKE_Y = test[\'model\'].map(LAB2I).to_numpy(dtype=int)\nelse:\n    DATA_DIR = next(p for p in [pathlib.Path(\'/kaggle/input/competitions/name-that-llm\'), pathlib.Path(\'/kaggle/input/name-that-llm\')]\n                    if (p / \'train.csv\').exists())\n    train, test = pd.read_csv(DATA_DIR / \'train.csv\'), pd.read_csv(DATA_DIR / \'test.csv\')\n    if os.environ.get(\'NTL_FAKE_TEST\'):   # timing probe: pool passages as a stand-in test (same rows/seed as infer.py --fake_test)\n        test = pd.read_parquet(os.environ[\'NTL_POOL\']).sample(int(os.environ[\'NTL_FAKE_TEST\']), random_state=0)[[\'text\']].reset_index(drop=True)\n        test[\'id\'] = [f\'fake{i}\' for i in range(len(test))]\ntest[\'text\'] = test[\'text\'].fillna(\'\').astype(str)\nTR, TE = train[\'text\'].tolist(), test[\'text\'].tolist()\ny = train[\'model\'].map(LAB2I).to_numpy(dtype=int)\nPROMPT = train[\'prompt_id\'].to_numpy(dtype=object)\nTINY = len(TE) < 50   # the interactive 3-row test: exercise the group paths on training rows so the hidden rerun cannot hit untested code\nDEV = \'cuda\' if torch is not None and torch.cuda.is_available() else \'cpu\'\nprint(f\'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}\', flush=True)\n\nLOGIT = np.zeros((len(TE), K))           # the running log-probability blend\n\n\ndef add_member(key, prob, rows=None, weight=None):\n    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""\n    w = W[key] if weight is None else weight\n    lp = np.log(np.clip(prob, 1e-9, 1.0))\n    if rows is None:\n        LOGIT[:] += w * lp\n    else:\n        LOGIT[rows] += w * lp\n    print(f\'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)\', flush=True)\n\n\ndef softmax_rows(S):\n    S = S - S.max(1, keepdims=True)\n    P = np.exp(S)\n    return P / P.sum(1, keepdims=True)\n\n\ndef full_prob(P_part, classes):\n    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""\n    P = np.zeros((len(P_part), K))\n    P[:, classes] = P_part\n    return P\n\n\ndef group_centre_dense(X, groups):\n    X = np.asarray(X, dtype=np.float32).copy()\n    for g in np.unique(groups):\n        i = groups == g\n        X[i] -= X[i].mean(0)\n    return X\n\n\ndef self_test_rows():\n    """Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny)."""\n    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]\n    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]\n\n# ============================================================ 1. grouping the test pool\ndef average_linkage_groups(D, thresh):\n    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""\n    if len(D) < 2:\n        return np.ones(len(D), dtype=int)\n    D = np.asarray(D, dtype=np.float64).copy()\n    np.fill_diagonal(D, 0.0)\n    return fcluster(linkage(squareform(D, checks=False), method=\'average\'), thresh, criterion=\'distance\')\n\n\ndef content_cosine_distance(texts, max_features=None):\n    X = TfidfVectorizer(stop_words=\'english\', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)\n    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)\n\n\ndef mixed_groups(texts, E, thresh=GROUP_THRESH):\n    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""\n    if len(texts) < 2:\n        return np.ones(len(texts), dtype=int)\n    Z = np.asarray(E, dtype=np.float64)\n    Z = Z - Z.mean(0)\n    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12\n    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))\n    return average_linkage_groups(D, thresh)\n\n\ndef normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):\n    """The stylegram\'s grouping: content distance divided by its median off-diagonal value, cut at thresh."""\n    if len(texts) < 2:\n        return np.ones(len(texts), dtype=int)\n    D = content_cosine_distance(texts, max_features=100_000)\n    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)\n    return average_linkage_groups(D / med, thresh)\n\n\ndef eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):\n    size = np.bincount(groups)[groups]\n    return (size >= lo) & (size <= hi)\n\n\ndef describe_groups(name, groups):\n    size = np.bincount(groups)[groups]\n    print(f\'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}\'\n          f\'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}\', flush=True)\n\n\nG_TE, GROUP_ROWS = None, None   # set after the GLM pass (section 4)\n\n# ============================================================ 2. style n-grams + extra passages + gates\nFUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "\n                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "\n                           "without for of by over under between through into about is are was were be been being have has had do does did "\n                           "can could will would shall should may might must not no nor all any some each every both either neither other "\n                           "another such only also just very too more most less least here there when where why how what whether then now "\n                           "still yet even much many few little".split())\nTOKEN_RE = re.compile(r"\\w+|[^\\w\\s]")\n\n\ndef shape_view(text):\n    return re.sub(r"[^\\W\\d_]+", lambda m: \'X\' if m[0].isupper() else \'x\', text)\n\n\ndef function_view(text):\n    return \' \'.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else \'CONTENT\' for t in TOKEN_RE.findall(text))\n\n\nclass StyleGrams:\n    """Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text)."""\n    VIEWS = [(\'char\', str, (3, 5), 250_000, 1.0), (\'word\', str, (1, 3), 180_000, 0.7),\n             (\'char\', shape_view, (2, 6), 60_000, 0.5), (\'word\', function_view, (1, 3), 50_000, 0.5)]\n\n    def fit(self, texts):\n        self.vecs = []\n        for analyzer, view, ngr, maxf, w in self.VIEWS:\n            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)\n            if analyzer == \'word\':\n                opts[\'token_pattern\'] = r"(?u)\\b\\w+\\b|[^\\w\\s]"\n            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))\n        return self\n\n    def transform(self, texts):\n        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format=\'csr\')\n\n\ndef sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):\n    """Subtract each group\'s mean row (sparse); groups outside [lo, hi] are left as they are."""\n    X = csr_matrix(X)\n    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)\n    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))\n    if not len(keep):\n        return X.copy()\n    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))\n    rows = np.flatnonzero(col[inv] >= 0)\n    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))\n    means = diags(1.0 / cnt[keep]) @ (M.T @ X)\n    return (X - M @ means).tocsr()\n\n\ndef svc_probs(svc, X):\n    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)\n\n\n# ---- public in-style passages (prompt-bank protocol), one per (prompt, label); prompts answered by >= EXTRA_MIN_LABELS labels\ndef load_extra_passages():\n    sources = [(\'**/ntl_extra_passages_claude.csv\', \'model\', None), (\'**/gen_passages.csv\', \'model\', None),\n               (\'**/ntl_generations.csv\', \'label\', \'finish_reason\'), (\'**/ntl_proxy_generations.csv\', \'label\', None),\n               (\'**/ntl_generated*.csv\', \'model\', \'finish\'), (\'**/passages_clean.csv\', \'model\', None)]\n    parts = []\n    for pattern, label_col, finish_col in sources:\n        for f in sorted(glob.glob(\'/kaggle/input/\' + pattern, recursive=True)):\n            d = pd.read_csv(f)\n            if finish_col and finish_col in d:\n                d = d[d[finish_col].eq(\'stop\')]\n            d = d.rename(columns={label_col: \'model\'})[[\'prompt_id\', \'model\', \'text\']].copy()\n            d[\'prompt_id\'] = (\'pv_\' if f.endswith(\'passages_clean.csv\') else \'\') + d[\'prompt_id\'].astype(str)   # prvsiyan\'s prompts are a different bank\n            parts.append(d)\n    if not parts:\n        return pd.DataFrame(columns=[\'prompt_id\', \'model\', \'text\'])\n    x = pd.concat(parts, ignore_index=True)\n    x = x[x[\'model\'].isin(LABELS)].copy()\n    x[\'text\'] = x[\'text\'].astype(str).map(lambda t: re.sub(r\'\\s+\', \' \', re.sub(r\'^\\*\\*(.*)\\*\\*$\', r\'\\1\', t.strip())).strip())\n    x = x[x[\'text\'].str.split().str.len().between(30, 300)].drop_duplicates(subset=[\'model\', \'text\'])\n    x = x.drop_duplicates(subset=[\'prompt_id\', \'model\'])\n    x = x[x.groupby(\'prompt_id\')[\'model\'].transform(\'nunique\') >= EXTRA_MIN_LABELS].reset_index(drop=True)\n    return x\n\n\nEXTRA = load_extra_passages()\nEX_T, EX_Y, EX_G = EXTRA[\'text\'].tolist(), EXTRA[\'model\'].map(LAB2I).to_numpy(dtype=int), EXTRA[\'prompt_id\'].to_numpy(dtype=object)\nprint(f\'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts\', flush=True)\n\n\ndef extra_sample_weights():\n    """Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class\'s extra rows share EXTRA_SHARE of it."""\n    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)\n    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)\n    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)\n    return np.concatenate([w_tr, w_ex])\n\n\ngrams = StyleGrams().fit(TR + EX_T)                       # vocabulary and idf from train + extra\nX_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)\n\n\ndef fit_svc(X, labels, C=0.5, **kw):\n    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)\n\n\nif X_ex is not None:\n    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())\nelse:\n    svc_plain = fit_svc(X_tr, y)\nP_style = svc_probs(svc_plain, X_te)\n\n# centred twin: prompt-centred training rows (extra rows centred within their own prompts) -> inferred test groups\nX_tr_c = sparse_group_centre(X_tr, PROMPT)\nif X_ex is not None:\n    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),\n                          sample_weight=extra_sample_weights())\nelse:\n    svc_centred = fit_svc(X_tr_c, y)\nG_style = normalised_content_groups(TE)\nrows_style = eligible(G_style)\ndescribe_groups(\'stylegram groups\', G_style)\nif rows_style.any():\n    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))\nif TINY:   # exercise the centred path on training rows grouped by their true prompt (output discarded)\n    st_idx, st_g = self_test_rows()\n    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])\n    print(f\'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}\', flush=True)\nadd_member(\'stylegram\', P_style)\ndel X_tr_c, svc_plain, svc_centred, P_style\n\n\n# ---- gates: style SVCs that only re-rank inside one family after the blend\ndef family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):\n    g = StyleGrams().fit(texts)\n    svc = LinearSVC(C=C, class_weight=\'balanced\' if balanced else None, random_state=42, max_iter=4000).fit(\n        g.transform(texts), labels, sample_weight=sample_weight)\n    assert list(svc.classes_) == list(classes)\n    d = svc.decision_function(g.transform(TE))\n    return np.column_stack([-d, d]) if len(classes) == 2 else d\n\n\nGATES = {}\napi_claude = sorted(glob.glob(\'/kaggle/input/**/ntl_extra_passages_claude.csv\', recursive=True))\nif api_claude:\n    api = pd.read_csv(api_claude[0])\n    m = np.isin(y, FAMILY_IDX[\'claude\'])\n    GATES[\'claude\'] = family_gate([t for t, k in zip(TR, m) if k] + api[\'text\'].astype(str).tolist(),\n                                  np.concatenate([y[m], api[\'model\'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX[\'claude\'], 0.5)\n    print(f\'claude gate: {int(m.sum())} official + {len(api)} API rows\', flush=True)\n    del api\nelse:\n    print(\'Claude API passages not attached: Claude gate off\', flush=True)\nfor fam, C in [(\'gpt\', 1.0), (\'gemini\', 1.0), (\'glm\', 0.5)]:\n    m, me = np.isin(y, FAMILY_IDX[fam]), np.isin(EX_Y, FAMILY_IDX[fam])\n    GATES[fam] = family_gate([t for t, k in zip(TR, m) if k] + [t for t, k in zip(EX_T, me) if k], np.concatenate([y[m], EX_Y[me]]),\n                             FAMILY_IDX[fam], C, sample_weight=np.concatenate([np.ones(int(m.sum())), np.full(int(me.sum()), 0.25)]), balanced=True)\n    print(f\'{fam} gate: {int(m.sum())} official + {int(me.sum())} extra rows\', flush=True)\ndel grams, X_tr, X_ex, X_te\n\n# ============================================================ 3. stylometric features + LightGBM\nPUNCT = list(\',.;:!?()[]"\\\'-\') + [\'—\', \'–\', \'‘\', \'’\', \'“\', \'”\', \'…\', \'*\', \'/\', \'%\']\nSTOPS = sorted(ENGLISH_STOP_WORDS)\nSENT_RE, WORD_RE = re.compile(r\'(?<=[.!?])\\s+\'), re.compile(r"[A-Za-z’\']+")\n\n\ndef stylometry(t):\n    words = WORD_RE.findall(t)\n    low = [w.lower() for w in words]\n    nw, nc = max(len(words), 1), max(len(t), 1)\n    sents = [s for s in SENT_RE.split(t.strip()) if s]\n    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)\n    wl = np.array([len(w) for w in words] or [0], dtype=float)\n    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,\n         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,\n         t.count(\', \') / nw, t.count(\' and \') / nw, t.count(\' but \') / nw,\n         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in (\'this\', \'these\', \'it\', \'they\')) / max(len(sents), 1),\n         sum(w.endswith(\'ly\') for w in words) / nw, sum(w.endswith(\'ing\') for w in words) / nw, sum(w.endswith(\'tion\') for w in words) / nw]\n    f += [t.count(q) / nw for q in PUNCT]\n    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)\n    f += [counts.get(s, 0) / nw for s in STOPS]\n    return f\n\n\nF_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))\ntry:\n    import lightgbm as lgb\n    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,\n                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)\n    add_member(\'stylometry_lgb\', full_prob(booster.predict_proba(F_te), booster.classes_))\n    del booster\nexcept ImportError:\n    assert SMOKE, \'lightgbm missing\'\n    print(\'smoke: lightgbm unavailable, stylometry member skipped\', flush=True)\ndel F_tr, F_te\n\n# ============================================================ 4b. shared-prompt Gaussian group head\n# Verbatim core of our research tool tools/group_gaussian.py (sha256 8a42427b77a37f56...), numpy only.\n# Model per dense view, in a reduced space z (standardise -> within-prompt-residual PCA, fitted on training rows):\n#   z_i = mu_{y_i} + p_g + e_i,  p_g ~ N(0, S_p) shared by every passage of prompt group g,  e_i ~ N(0, S_e).\n# A joint diagonalisation makes the dimensions independent; for a detected group of n passages the exact joint\n# log-likelihood of a label tuple is sum_i a_i(c_i) + sum_{i<j} b_ij(c_i, c_j). Groups of 2-3 are decoded exactly over\n# all label tuples with the at-most-once constraint; larger groups by a cavity EM with the finite-population-corrected\n# prompt-effect estimate lam / (1 + (n-1) lam); singletons integrate the prompt effect out (never used here).\nimport itertools\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n# =================================================================== primitives ====\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n\n# ============================================================ 4. frozen models: plain heads, centred twins, Gaussian group heads\nif not SMOKE:\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\nCACHED_PARTS = {}   # key -> {part name: (train features, test features)}\n\n\ndef find_checkpoint(model_type, hidden=None, dir_name=None):\n    """A checkpoint directory under /kaggle/input by model_type (+ hidden size) or by directory name."""\n    pattern = f\'/kaggle/input/**/{dir_name}/config.json\' if dir_name else \'/kaggle/input/**/config.json\'\n    for cfg in sorted(glob.glob(pattern, recursive=True)):\n        try:\n            c = json.load(open(cfg))\n        except Exception:\n            continue\n        if dir_name or (c.get(\'model_type\') == model_type and (hidden is None or c.get(\'hidden_size\') == hidden)):\n            return os.path.dirname(cfg)\n    return None\n\n\ndef load_truncated_decoder(path, keep, dtype):\n    cfg = AutoConfig.from_pretrained(path)\n    cfg.num_hidden_layers = keep\n    if getattr(cfg, \'layer_types\', None):\n        cfg.layer_types = cfg.layer_types[:keep]\n    kw = dict(config=cfg, dtype=dtype)\n    if DEV == \'cuda\':\n        kw[\'device_map\'] = \'auto\'\n    return AutoModel.from_pretrained(path, **kw).eval()\n\n\ndef style_token_mask(tok):\n    """Vocabulary mask of \'style\' tokens: the decoded piece is an English stop word or pure punctuation."""\n    pieces = [tok.decode([i]) for i in range(len(tok))]\n    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r\'\\s*[^\\w\\s]+\\s*\', p)) for p in pieces])\n\n\n@torch.no_grad()\ndef pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):\n    """{\'L{layer}_{pool}\': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""\n    H = model.config.hidden_size\n    out = {f\'L{L}_{pool}\': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}\n    smask = style_token_mask(tok) if any(pool == \'style\' for _, pool in parts) else None\n    dev = next(model.parameters()).device\n    order = np.argsort([len(t) for t in texts])\n    retried = 0\n    for b0 in range(0, len(texts), bs):\n        idx = order[b0:b0 + bs]\n        if decoder:\n            batch = [(tok.bos_token or \'\') + texts[i] for i in idx]\n            enc = tok(batch, return_tensors=\'pt\', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)\n        else:\n            enc = tok([texts[i] for i in idx], return_tensors=\'pt\', padding=True, truncation=True, max_length=max_len).to(dev)\n        hs = model(**enc, output_hidden_states=True).hidden_states\n        if decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):   # fp16 overflow: redo the batch in bf16\n            retried += 1\n            model.to(torch.bfloat16)\n            hs = model(**enc, output_hidden_states=True).hidden_states\n        m_all = enc[\'attention_mask\'].float()\n        for L, pool in parts:\n            m = m_all * smask.to(m_all.device)[enc[\'input_ids\']].float() if pool == \'style\' else m_all\n            m = m.unsqueeze(-1).to(hs[L].device)\n            out[f\'L{L}_{pool}\'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()\n        del hs, enc\n    if retried:\n        print(f\'  {retried} batches redone in bf16\', flush=True)\n    return {k: np.nan_to_num(v) for k, v in out.items()}\n\n\ndef pseudo_parts(parts, dim=48):\n    """SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code."""\n    from sklearn.decomposition import TruncatedSVD\n    tf = TfidfVectorizer(analyzer=\'char\', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)\n    out = {}\n    for j, (L, pool) in enumerate(parts):\n        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))\n        out[f\'L{L}_{pool}\'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))\n    return out\n\n\ndef extract_parts(key, parts, loader):\n    """Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size)."""\n    if SMOKE:\n        return pseudo_parts(parts)\n    model, tok, decoder, bs = loader()\n    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)\n    del model, tok\n    gc.collect()\n    if DEV == \'cuda\':\n        torch.cuda.empty_cache()\n    return {k: (tr[k], te[k]) for k in tr}\n\n\ndef decoder_loader(model_type, hidden, keep):\n    def load():\n        path = find_checkpoint(model_type, hidden)\n        assert path, f\'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input\'\n        tok = AutoTokenizer.from_pretrained(path)\n        tok.padding_side = \'right\'\n        if tok.pad_token is None:\n            tok.pad_token = tok.eos_token\n        return load_truncated_decoder(path, keep, torch.float16 if DEV == \'cuda\' else torch.float32), tok, True, 8\n    return load\n\n\ndef encoder_loader(dir_name):\n    def load():\n        path = find_checkpoint(None, dir_name=dir_name)\n        assert path, f\'encoder directory {dir_name} not found under /kaggle/input\'\n        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16\n    return load\n\n\ndef concat_parts(cache, parts):\n    return (np.concatenate([cache[f\'L{L}_{p}\'][0] for L, p in parts], 1), np.concatenate([cache[f\'L{L}_{p}\'][1] for L, p in parts], 1))\n\n\ndef plain_and_twin(key, Etr, Ete, C):\n    """Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows."""\n    if W[key] > 0:\n        sc = StandardScaler().fit(Etr)\n        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)\n        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)\n        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))\n    else:\n        print(f\'  {key}: plain head has weight 0 in the 0.563 blend, not fitted\', flush=True)\n    if key in TWIN_W:\n        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))\n        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)\n        if GROUP_ROWS.any():\n            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))\n            add_member(key + \'_twin\', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])\n        if TINY:\n            st_idx, st_g = self_test_rows()\n            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))\n            print(f\'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}\', flush=True)\n\n\ndef gaussian_group_head(name, spec, Xtr, Xte):\n    """Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group."""\n    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec[\'d\']))\n    prm[\'T1\'], prm[\'Tg\'] = spec[\'T1\'], spec[\'Tg\']\n    print(f\'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}\', flush=True)\n    if GROUP_ROWS.any():\n        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)\n        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec[\'w\'])\n    if TINY:\n        st_idx, st_g = self_test_rows()\n        LP = scores(Xtr[st_idx], st_g, prm)\n        print(f\'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}\', flush=True)\n\n\nfor key, model_type, hidden, parts, extra_parts, C in LM_HEADS:\n    print(f\'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}\', flush=True)\n    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))\n    if G_TE is None:   # the GLM pass comes first: its layer-24 pool defines the test groups\n        G_TE = mixed_groups(TE, cache[f\'L{GROUP_LAYER}_mean\'][1])\n        GROUP_ROWS = eligible(G_TE)\n        describe_groups(\'test groups (content + GLM L24)\', G_TE)\n        if SMOKE:\n            same_g, same_p = G_TE[:, None] == G_TE[None, :], test[\'prompt_id\'].to_numpy(dtype=object)[:, None] == test[\'prompt_id\'].to_numpy(dtype=object)[None, :]\n            off = ~np.eye(len(TE), dtype=bool)\n            print(f\'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} \'\n                  f\'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}\', flush=True)\n    CACHED_PARTS[key] = cache\n    plain_and_twin(key, *concat_parts(cache, parts), C)\n    for name, spec in GAUSS_HEADS.items():\n        if spec[\'src\'] == key:\n            gaussian_group_head(name, spec, *cache[spec[\'part\']])\n    del cache\n    CACHED_PARTS.pop(key)\n    gc.collect()\n\nfor key, dir_name, parts, C in ENC_HEADS:\n    print(f\'[{key}] {dir_name} parts {parts}\', flush=True)\n    cache = extract_parts(key, parts, encoder_loader(dir_name))\n    plain_and_twin(key, *concat_parts(cache, parts), C)\n    for name, spec in GAUSS_HEADS.items():\n        if spec[\'src\'] == key:\n            gaussian_group_head(name, spec, *cache[spec[\'part\']])\n    del cache\n    gc.collect()\n\n# ============================================================ 5. LoRA classifiers (inference only)\n@torch.no_grad()\ndef score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):\n    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;\n    within-Claude readout -> Claude gate."""\n    import importlib.util as iu\n    _find = iu.find_spec   # Kaggle\'s torchao build is rejected by peft; hide it (unused here)\n    iu.find_spec = lambda name, *a, **k: None if name.split(\'.\')[0] == \'torchao\' else _find(name, *a, **k)\n    from peft import PeftModel\n    import peft.import_utils as pi\n    pi.is_torchao_available = lambda *a, **k: False\n    from safetensors.torch import load_file\n    model_type, H, keep = trunk_spec\n    base = find_checkpoint(model_type, H)\n    assert base, f\'no {model_type} base checkpoint under /kaggle/input\'\n    cfg = AutoConfig.from_pretrained(base)\n    cfg.num_hidden_layers, cfg.use_cache = keep, False\n    if getattr(cfg, \'layer_types\', None) is not None:\n        cfg.layer_types = cfg.layer_types[:keep]\n    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map=\'auto\').eval()\n    for key, marker, norms_file, anchor_file in checkpoints:\n        if key == \'lora24_public\' and os.environ.get(\'MERGE26_PUBLIC24\'):\n            hits = [str(pathlib.Path(os.environ[\'MERGE26_PUBLIC24\'])/\'COMPLETE.json\')]\n            assert pathlib.Path(hits[0]).is_file()\n        else:\n            hits = sorted(glob.glob(\'/kaggle/input/\' + marker, recursive=True))\n        if not hits:   # upstream training kernel no longer public: drop this member, keep the rest of the stack\n            print(f\'  SKIP {key}: checkpoint not attached ({marker})\', flush=True); continue\n        ck = pathlib.Path(os.path.dirname(hits[0]))\n        print(f\'[{key}] {ck}\', flush=True)\n        tok = AutoTokenizer.from_pretrained(ck / \'tokenizer\')\n        tok.padding_side = \'right\'\n        if tok.pad_token is None:\n            tok.pad_token = tok.eos_token\n        lm = PeftModel.from_pretrained(trunk, ck / \'adapter\').eval()\n        state = load_file(str(ck / norms_file))\n        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])\n        norms.load_state_dict({k[len(\'norms.\'):]: v for k, v in state.items() if k.startswith(\'norms.\')})\n        ro = np.load(ck / \'readouts.npz\')\n        assert list(ro[\'classes\']) == LABELS and list(ro[\'pool_layers\']) == pool_layers\n        dev = lm.get_input_embeddings().weight.device\n        norms = norms.to(dev).float()\n\n        def token_ids(texts):\n            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)[\'input_ids\']]   # these checkpoints\' anchors match the tokenizer\'s own eos\n\n        def features(texts, bs):\n            ids = token_ids(texts)\n            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)\n            order = np.argsort([len(t) for t in ids], kind=\'stable\')\n            for b0 in range(0, len(order), bs):\n                idx = order[b0:b0 + bs]\n                batch = tok.pad({\'input_ids\': [ids[i] for i in idx]}, return_tensors=\'pt\').to(dev)\n                with torch.autocast(\'cuda\', dtype=torch.float16):\n                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states\n                m = batch[\'attention_mask\'].unsqueeze(-1).float()\n                m[:, 0] = 0   # the first token is masked out of the mean, as in training\n                Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))\n                                    for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()\n                del hs, batch\n            return Z\n\n        if (ck / anchor_file).exists():   # the checkpoint\'s own features for training rows 0..2: our forward pass must reproduce them\n            saved = np.load(ck / anchor_file)[\'features\'].astype(np.float32)\n            ours = features(TR[:3], 3)\n            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))\n            print(f\'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}\', flush=True)\n            assert rel < 0.02, f\'{key}: our features do not match the checkpoint\'\n        Z = features(TE, LORA_BS)\n        if key == \'lora24_public\' and os.environ.get(\'MERGE26_SPECTRAL_FEATURES\'):\n            import hashlib\n            from pathlib import Path\n            expected = \'a6aff409ae2df897717875373d038635fe5012fbf93557c90aea3457fda0e7f8\'\n            assert hashlib.sha256((ck/\'adapter/adapter_model.safetensors\').read_bytes()).hexdigest()==expected\n            assert hashlib.sha256((ck/norms_file).read_bytes()).hexdigest()==\'2176ade29e6eb97bd3a6c204f944959f9757a80c6f1ff475898c23bce60ecd96\'\n            assert list(pool_layers)==[2,4,6,8,12,16,20,24] and Z.shape==(len(TE),32768)\n            raw=Path(os.environ[\'MERGE26_SPECTRAL_FEATURES\'])\n            assert raw.resolve().is_relative_to(Path(\'/tmp\'))\n            np.save(raw,Z,allow_pickle=False)\n            raw.with_suffix(\'.json\').write_text(json.dumps(dict(ids=test.id.astype(str).tolist(),\n                labels=LABELS,layers=pool_layers,shape=list(Z.shape),dtype=str(Z.dtype),\n                checkpoint_metadata=str(ck/\'COMPLETE.json\'))))\n            print(\'MERGE26_MATCHED_SPECTRAL_FEATURES\',len(Z),flush=True)\n            if os.environ.get(\'MERGE26_PUBLIC24_EXPORT_ONLY\')==\'1\':\n                trunk=lm.unload()\n                del lm,Z\n                gc.collect();torch.cuda.empty_cache()\n                continue\n        add_member(key, softmax_rows(Z @ ro[\'plain_coef\'].T + ro[\'plain_intercept\']))\n        if \'claude_coef\' in ro.files:\n            lc = Z @ ro[\'claude_coef\'].T + ro[\'claude_intercept\']\n            GATES[\'claude_lora\'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)\n        if key in LORA_TWIN_W and GROUP_ROWS.any():\n            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])\n            add_member(key + \'_twin\', softmax_rows(Zc @ ro[\'center_coef\'].T + ro[\'center_intercept\']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])\n        trunk = lm.unload()\n        del lm, Z\n        gc.collect()\n        torch.cuda.empty_cache()\n    del trunk\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\nif not SMOKE:\n    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:\n        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)\n\n# ============================================================ 6. token-attention heads + UltraBERT\nclass TokenAttention(torch.nn.Module if torch is not None else object):   # the module of the public training kernels (lucasboesen/name-that-llm-train-tokattn, -tokattn2)\n    def __init__(self, k, h=4096):\n        super().__init__()\n        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)\n\n    def forward(self, x, m):\n        h = torch.tanh(self.proj(self.drop(x)))\n        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)\n        mf = m.unsqueeze(-1).float()\n        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))\n\n\n@torch.no_grad()\ndef token_attention_probs(texts, head_file, backbone, bs=8):\n    hits = sorted(glob.glob(\'/kaggle/input/**/\' + head_file, recursive=True))\n    assert hits, f\'training-kernel output holding {head_file} not attached\'\n    d = torch.load(hits[0], map_location=\'cpu\')\n    assert list(d[\'classes\']) == LABELS\n    decoder = d.get(\'decoder\', True)\n    path = find_checkpoint(\'qwen3_5\') if backbone == \'qwen3_5\' else find_checkpoint(None, dir_name=\'deberta-v3-large\')\n    assert path, f\'backbone {backbone} not attached\'\n    cfg = AutoConfig.from_pretrained(path)\n    if decoder:\n        tc = getattr(cfg, \'text_config\', None) or cfg\n        tc.num_hidden_layers = d[\'keep\']\n        if getattr(tc, \'layer_types\', None):\n            tc.layer_types = tc.layer_types[:d[\'keep\']]\n        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map=\'auto\').eval()\n    else:\n        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()\n    tok = AutoTokenizer.from_pretrained(path)\n    tok.padding_side = \'right\'\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    H = d.get(\'hidden\', 4096)\n    net = TokenAttention(K, H)\n    net.load_state_dict(d[\'state_dict\'])\n    dev = lm.get_input_embeddings().weight.device\n    net, mu, sd = net.to(dev).eval(), d[\'mu\'].float().to(dev), d[\'sd\'].float().to(dev)\n    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d[\'max_len\'])[\'input_ids\']\n    P = np.zeros((len(texts), K))\n    order = np.argsort([len(t) for t in ids])\n    for b0 in range(0, len(order), bs):\n        idx = order[b0:b0 + bs]\n        batch = tok.pad({\'input_ids\': [ids[i] for i in idx]}, return_tensors=\'pt\').to(dev)\n        out = lm(input_ids=batch[\'input_ids\'], attention_mask=batch[\'attention_mask\'], output_hidden_states=True)\n        hs = (out.hidden_states if getattr(out, \'hidden_states\', None) is not None else out.language_model_outputs.hidden_states)[d[\'layer\']]\n        n = batch[\'attention_mask\'].sum(1)\n        T = max(int(n.max()) - 1, 1)\n        x = torch.zeros(len(idx), T, H, device=dev)\n        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)\n        for r in range(len(idx)):   # first token dropped, states rounded through fp16, as in training\n            k = int(n[r]) - 1\n            if k > 0:\n                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)\n                m[r, :k] = True\n        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()\n        del hs, out, batch\n    del lm\n    gc.collect()\n    torch.cuda.empty_cache()\n    return P\n\n\n@torch.no_grad()\ndef ultrabert_probs(texts):\n    from transformers import AutoModelForSequenceClassification\n    folds = None\n    for root in glob.glob(\'/kaggle/input/**/ultrabert-380m-ntl\', recursive=True):\n        fs = [f for f in sorted(glob.glob(os.path.join(root, \'fold*\'))) if os.path.exists(os.path.join(f, \'config.json\'))\n              and json.load(open(os.path.join(f, \'config.json\'))).get(\'model_type\') == \'ultrabert\']\n        if len(fs) == 5:\n            folds = fs\n            break\n    assert folds, \'ultrabert-380m-ntl (5 folds) not attached\'\n    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)\n    ids = tok(texts, truncation=True, max_length=512)[\'input_ids\']\n    order = np.argsort([len(e) for e in ids])\n    P = np.zeros((len(texts), K))\n    for fp in folds:\n        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation=\'sdpa\').to(DEV).eval()\n        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]\n        for b0 in range(0, len(order), 32):\n            idx = order[b0:b0 + 32]\n            b = tok.pad({\'input_ids\': [ids[j] for j in idx]}, return_tensors=\'pt\').to(DEV)\n            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)\n        del em\n        gc.collect()\n        torch.cuda.empty_cache()\n    return P\n\n\nif not SMOKE:\n    for key, head_file, backbone in TOKATTN_HEADS:\n        if TINY:   # in-sample check on training rows the head was fitted on: must be high, or the checkpoint is not being reproduced\n            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))\n            print(f\'  {key} in-sample check on 40 training rows: acc {acc:.2f}\', flush=True)\n            assert acc > 0.5, f\'{key} does not reproduce its training fit\'\n        add_member(key, token_attention_probs(TE, head_file, backbone))\n    add_member(\'ultrabert\', ultrabert_probs(TE))\n\n\n\nnp.save(\'/kaggle/working/comp_logit.npy\', LOGIT)\njson.dump({\'labels\': LABELS, \'ids\': test[\'id\'].astype(str).tolist(), \'seconds\': time.time() - T0}, open(\'/kaggle/working/comp_meta.json\', \'w\'))\nprint(f\'COMP STACK DONE {time.time() - T0:.0f}s\', flush=True)\n')
    (root/'fulltext_pooling.py').write_text('"""Keep eight pooled readouts, without retaining every layer\'s token states."""\nfrom contextlib import nullcontext\nimport torch\nfrom torch import nn\n\nclass MeanPoolHooks:\n    def __init__(self,model,depths,norms,attention_mask):\n        assert isinstance(model.norm,nn.Identity)\n        assert len(depths)==len(norms)\n        self.model,self.depths,self.norms=model,list(depths),norms\n        self.mask=attention_mask.unsqueeze(-1).float();self.den=self.mask.sum(1)\n        assert (self.den>0).all()\n        self.values,self.handles={},[]\n    def hook_for(self,depth,norm):\n        def hook(module,inputs,output):\n            assert isinstance(output,torch.Tensor) and output.ndim==3\n            assert output.shape[:2]==self.mask.shape[:2] and depth not in self.values\n            context=torch.autocast(\'cuda\',enabled=False) if output.is_cuda else nullcontext()\n            with context:self.values[depth]=norm((output.float()*self.mask).sum(1)/self.den)\n        return hook\n    def __enter__(self):\n        for depth,norm in zip(self.depths,self.norms):\n            assert 1<=depth<=len(self.model.layers)\n            self.handles.append(self.model.layers[depth-1].register_forward_hook(self.hook_for(depth,norm)))\n        return self\n    def __exit__(self,*args):\n        for handle in self.handles:handle.remove()\n        self.handles.clear()\n    def features(self):\n        assert set(self.values)==set(self.depths)\n        return torch.cat([self.values[depth] for depth in self.depths],dim=-1)\n')
    (root/'fusion.py').write_text('"""One probability mixture BEFORE the existing nlm balancing/postprocess.\n\nCredits: Dhruv Tibarewal nlm-v27; lightningv08 fulltext stack;\nShaksham Karkiii VARIANT4 shares; Kirill Tymchishin old4404 readouts.\n"""\nimport numpy as np\n\nSHARES={\'nlm\':0.7225,\'spectral4404\':0.1275,\'fresh32\':0.15}\n\n\ndef softmax(logits):\n    x=np.asarray(logits,dtype=np.float64)\n    assert x.ndim==2 and np.isfinite(x).all()\n    x=np.exp(x-x.max(axis=1,keepdims=True))\n    return x/x.sum(axis=1,keepdims=True)\n\n\ndef align(ids,labels,source_ids,source_labels,values):\n    ids=list(map(str,ids));labels=list(map(str,labels))\n    source_ids=list(map(str,source_ids));source_labels=list(map(str,source_labels))\n    assert len(ids)==len(set(ids))==len(source_ids)==len(set(source_ids))\n    assert set(ids)==set(source_ids),\'Missing, extra or mismatched test IDs\'\n    assert len(labels)==len(set(labels))==len(source_labels)==len(set(source_labels))==18\n    assert set(labels)==set(source_labels),\'Wrong model-label vocabulary\'\n    values=np.asarray(values,dtype=np.float64)\n    assert values.shape==(len(source_ids),len(source_labels))\n    assert np.isfinite(values).all()\n    row={v:i for i,v in enumerate(source_ids)}\n    col={v:i for i,v in enumerate(source_labels)}\n    return values[np.ix_([row[i] for i in ids],[col[l] for l in labels])]\n\n\ndef probability(values):\n    p=np.asarray(values,dtype=np.float64)\n    assert p.ndim==2 and p.shape[1]==18 and np.isfinite(p).all()\n    assert (p>=0).all() and np.allclose(p.sum(axis=1),1,atol=1e-5)\n    return p/p.sum(axis=1,keepdims=True)\n\n\ndef combine(nlm,spectral4404,fresh32,shares=None):\n    shares=SHARES if shares is None else shares\n    assert set(shares)==set(SHARES) and all(w>=0 for w in shares.values())\n    assert abs(sum(shares.values())-1)<1e-12\n    parts={\'nlm\':probability(nlm),\'spectral4404\':probability(spectral4404),\n           \'fresh32\':probability(fresh32)}\n    assert len({p.shape for p in parts.values()})==1\n    return probability(sum(shares[k]*parts[k] for k in parts))\n\n\ndef add_to_nlm(ids,labels,base,paths):\n    """Called once after nlm ATLAS and before its first Sinkhorn iteration."""\n    parts={}\n    for name in [\'spectral4404\',\'fresh32\']:\n        z=np.load(paths[name],allow_pickle=False)\n        values=softmax(z[\'logits\']) if name==\'fresh32\' else probability(z[\'probabilities\'])\n        parts[name]=align(ids,labels,z[\'ids\'],z[\'labels\'],values)\n    return combine(base,parts[\'spectral4404\'],parts[\'fresh32\'])\n')
    (root/'gauss.py').write_text('"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).\nTwo-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""\nimport itertools\nimport numpy as np\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n# =================================================================== primitives ====\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n')
    (root/'infer.py').write_text('"""End-to-end attribution pipeline (self-contained; runs on Kaggle T4x2 or locally in --sim mode).\n1. Mean-pooled hidden states from frozen LMs for train+test (computed here so precision/quantisation match).\n2. Per model: LR on raw + LR on prompt-centred features (train centred by prompt_id, test by estimated cluster).\n3. TF-IDF (char_wb + word/punct) prompt-centred LR.\n4. Test prompts recovered by size-capped average-linkage clustering on content-word TF-IDF.\n5. Sum log-probs, then Hungarian assignment inside each cluster (each model answers each prompt at most once).\n"""\nimport os, sys, time, gc, argparse, json\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport numpy as np, pandas as pd, scipy.sparse as sp, torch\nfrom scipy.optimize import linear_sum_assignment\n\nLABELS = [\'claude-fable-5\',\'claude-opus-4-6\',\'claude-opus-4-8\',\'claude-opus-5\',\'claude-sonnet-5\',\n \'gemini-3.1-pro-preview\',\'gemini-3.6-flash\',\'gemini-3.7-flash\',\'gemini-3.8-flash\',\'glm-5.3\',\'glm-5.3-flash\',\n \'gpt-5.5\',\'gpt-5.6-luna\',\'gpt-5.6-sol\',\'gpt-5.6-terra\',\'gpt-6-astra\',\'kimi-k3\',\'qwen3.8-max\']\nL2I = {l: i for i, l in enumerate(LABELS)}\n\np = argparse.ArgumentParser()\np.add_argument(\'--data\', default=\'/kaggle/input/competitions/name-that-llm\')\np.add_argument(\'--models\', default=\'\')   # json list of {"path":..,"layers":[fractions],"quant":"fp16|4bit|8bit","bs":16}\np.add_argument(\'--comp_path\', default=\'\'); p.add_argument(\'--comp_w\', type=float, default=0.0)\np.add_argument(\'--fake_test\', type=int, default=0, help=\'timing probe: score N pool passages as a stand-in test\')\np.add_argument(\'--atlas_probe\', type=int, default=0, help=\'timing probe: stop ATLAS after N passages\')\np.add_argument(\'--sim\', type=str, default=\'\', help=\'comma list of folds held out of train.csv as fake test\')\np.add_argument(\'--out\', default=\'submission.parquet\')\np.add_argument(\'--cache\', default=\'cache\')\np.add_argument(\'--dtype\', default=\'fp16\')\np.add_argument(\'--C\', type=float, default=0.005)\np.add_argument(\'--tfidf_w\', type=float, default=1.0)\np.add_argument(\'--distort_w\', type=float, default=0.0, help=\'weight of topic-masked (text distortion) TF-IDF member\')\np.add_argument(\'--adapt_thr\', type=float, default=0.15, help=\'cluster cut (avg cosine sim on content-word TF-IDF)\')\np.add_argument(\'--adapt_w\', type=float, default=0.0, help=\'weight of centred twin evidence on rows in clusters of size>=2 (0=off)\')\np.add_argument(\'--post\', type=int, default=0, help=\'1: spaced em-dash guard + Hungarian on dense test clusters (12-18)\')\np.add_argument(\'--atlas_dir\', default=\'\'); p.add_argument(\'--atlas_base\', default=\'\'); p.add_argument(\'--atlas_w\', default=\'0\', help=\'mix weight per bundle (comma list or one value)\')\np.add_argument(\'--atlas_margin\', type=float, default=0.0, help=\'>0: score ATLAS only on rows whose top-2 margin is below this\')\np.add_argument(\'--atlas_bs\', type=int, default=4); p.add_argument(\'--atlas_chunk\', type=int, default=3); p.add_argument(\'--atlas_budget\', type=float, default=8.3 * 3600)\np.add_argument(\'--sk_div\', type=int, default=1, help=\'1: divide blend by #members before Sinkhorn (ours); 0: raw blend (public notebook)\')\np.add_argument(\'--sinkhorn\', type=float, default=0.0, help=\'alpha of Sinkhorn class-marginal balancing (0=off)\')\np.add_argument(\'--sim_k\', default=\'\', help="sim only: keep \'lo-hi\' random passages per held-out prompt")\np.add_argument(\'--extra\', default=\'\', help=\'parquet/csv (text, model) of matched external passages, used by TF-IDF/distort members\')\np.add_argument(\'--extra_w\', type=float, default=0.5)\np.add_argument(\'--svc_w\', type=float, default=0.0, help=\'4-view n-gram LinearSVC on train + --pool\')\np.add_argument(\'--pool\', default=\'\')\np.add_argument(\'--save_lp\', default=\'\', help=\'also save normalised blend log-probs (n x 18, LABELS order) to this .npy\')\np.add_argument(\'--perpassage\', type=int, default=1, help=\'1: test passages treated independently (no clustering/centring/assignment)\')\na, _ = p.parse_known_args()\nos.makedirs(a.cache, exist_ok=True)\ntorch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False   # fp32 sums inside fp16 matmuls (free on T4)\nT0 = float(os.environ.get(\'MERGE26_STARTED_WALL\',time.time()))\ndef log(*x): print(f\'[{time.time()-T0:7.0f}s]\', *x, flush=True)\n\n# ---------------- data\ntrain = pd.read_csv(os.path.join(a.data, \'train.csv\'))\nif a.sim:\n    rng = np.random.RandomState(0); pids = np.array(sorted(train.prompt_id.unique())); rng.shuffle(pids)\n    pf = {q: i % 5 for i, q in enumerate(pids)}; fo = train.prompt_id.map(pf).values\n    hold = np.isin(fo, [int(x) for x in a.sim.split(\',\')])\n    if a.sim_k:\n        lo_, hi_ = map(int, a.sim_k.split(\'-\')); keep = np.zeros(len(train), bool); idx_h = np.where(hold)[0]\n        for g in np.unique(train.prompt_id.values[idx_h]):\n            ii = idx_h[train.prompt_id.values[idx_h] == g]; keep[rng.choice(ii, min(len(ii), rng.randint(lo_, hi_ + 1)), replace=False)] = True\n        hold = keep | (hold & False)\n    test = train[hold].drop(columns=[\'model\']).reset_index(drop=True); test_truth = train[hold].reset_index(drop=True)\n    hold_any = np.isin(fo, [int(x) for x in a.sim.split(\',\')])\n    train = train[~hold_any].reset_index(drop=True)\nelif a.fake_test:\n    test = pd.read_parquet(a.pool).sample(a.fake_test, random_state=0)[[\'text\']].reset_index(drop=True)\n    test[\'id\'] = [f\'fake{i}\' for i in range(len(test))]\nelse:\n    test = pd.read_csv(os.path.join(a.data, \'test.csv\'))\ntrain[\'text\'] = train.text.fillna(\'\'); test[\'text\'] = test.text.fillna(\'\')\ny = train.model.map(L2I).values; Gtr = train.prompt_id.values\nlog(\'train\', len(train), \'test\', len(test))\n\n# ---------------- utils\ndef center_groups(X, groups):\n    X = X.copy()\n    for g in np.unique(groups):\n        i = groups == g; X[i] -= X[i].mean(0)\n    return X\n\ndef torch_lr(Xtr, ytr, Xte, C, iters=200):\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    Xtr = torch.as_tensor(Xtr, dtype=torch.float32, device=dev); Xte = torch.as_tensor(Xte, dtype=torch.float32, device=dev)\n    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6; Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd\n    yt = torch.as_tensor(ytr, dtype=torch.long, device=dev)\n    W = torch.zeros(Xtr.shape[1], 18, device=dev, requires_grad=True); b = torch.zeros(18, device=dev, requires_grad=True)\n    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=iters, history_size=20, line_search_fn=\'strong_wolfe\')\n    def closure():\n        opt.zero_grad()\n        loss = torch.nn.functional.cross_entropy(Xtr @ W + b, yt, reduction=\'sum\') + 0.5 / C * (W ** 2).sum()\n        loss.backward(); return loss\n    opt.step(closure)\n    with torch.no_grad(): return torch.log_softmax(Xte @ W + b, -1).cpu().numpy()\n\ndef capped_agglo(S, cap=18, thr=0.02):\n    """Average linkage (Lance-Williams), never merging past `cap` members; stop when best avg sim < thr."""\n    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)\n    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)\n    while True:\n        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]\n        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)\n        if not np.isfinite(M[i, j]) or M[i, j] < thr: break\n        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])\n        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf\n        size[i] += size[j]; alive[j] = False; lab[lab == j] = i\n    return lab\n\ndef hungarian(lp, groups):\n    pred = lp.argmax(1).copy()\n    for g in np.unique(groups):\n        idx = np.where(groups == g)[0]; r, c = linear_sum_assignment(-lp[idx]); pred[idx[r]] = c\n    return pred\n\n# ---------------- 1. recover test prompts\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nif a.perpassage and a.adapt_w <= 0:\n    Gte = np.arange(len(test))\nelse:\n  try:\n    v = TfidfVectorizer(sublinear_tf=True, stop_words=\'english\', min_df=2, max_df=0.05)\n    Xc = v.fit_transform(test.text.values)\n    Gte = capped_agglo((Xc @ Xc.T).toarray(), 18, a.adapt_thr if a.perpassage else 0.02)\n  except ValueError as e:  # tiny interactive test file: nothing survives df pruning\n    log(\'clustering fallback:\', e); Gte = np.arange(len(test))\nlog(\'test clusters\', len(np.unique(Gte)), \'sizes\', np.bincount(np.bincount(Gte)[np.bincount(Gte) > 0]).tolist())\nif a.sim:\n    from sklearn.metrics import adjusted_rand_score\n    log(\'cluster ARI vs truth\', round(adjusted_rand_score(test_truth.prompt_id, Gte), 4))\n\nLP = {}\nEXT_T, EXT_Y = np.array([], dtype=object), np.array([], dtype=int)\nif a.extra:\n    _E = pd.read_parquet(a.extra) if a.extra.endswith(\'.parquet\') else pd.read_csv(a.extra)\n    _E = _E[_E.model.isin(LABELS)]; EXT_T = _E.text.astype(str).values; EXT_Y = _E.model.map(L2I).values\n    log(\'extra rows\', len(EXT_T))\nFIT_T = np.concatenate([train.text.values, EXT_T]); FIT_Y = np.concatenate([y, EXT_Y])\nFIT_W = np.concatenate([np.ones(len(train)), np.full(len(EXT_T), a.extra_w)])\n# ---------------- 2. TF-IDF centred LR\nfrom sklearn.linear_model import LogisticRegression\nmats_tr, mats_te = [], []\nif a.tfidf_w > 0:\n    for kw in [dict(analyzer=\'char_wb\', ngram_range=(2, 5), min_df=3, max_features=60000),\n               dict(token_pattern=r"\\w+|[^\\w\\s]", ngram_range=(1, 2), min_df=3, max_features=30000)]:\n        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T if a.perpassage else train.text.values)\n        mats_tr.append(vec.transform(FIT_T if a.perpassage else train.text.values)); mats_te.append(vec.transform(test.text.values))\n    if a.perpassage:\n        LP[\'tfidf\'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w\n    else:\n        Xtr = center_groups(sp.hstack(mats_tr).toarray().astype(np.float32), Gtr)\n        Xte = center_groups(sp.hstack(mats_te).toarray().astype(np.float32), Gte)\n        LP[\'tfidf\'] = LogisticRegression(C=5, max_iter=3000).fit(Xtr, y).predict_log_proba(Xte) * a.tfidf_w\n        del Xtr, Xte\ngc.collect(); log(\'tfidf done\')\nif a.svc_w > 0 and os.environ.get(\'MERGE26_SVC_BANK\'):\n    from svc_cache import predict_cache\n    LP[\'svc4\'] = predict_cache(os.environ[\'MERGE26_SVC_BANK\'],test.text.tolist(),LABELS)*a.svc_w\n    log(\'svc4 exact CPU prefit loaded\')\nelse:\n    if a.svc_w > 0:  # public 4-view n-gram LinearSVC (char / word+punct / shape / function words) on train + pool\n        import re as _re\n        from scipy.sparse import hstack as _hs\n        from sklearn.svm import LinearSVC\n        _FW = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())\n        _shape = lambda t: _re.sub(r"[^\\W\\d_]+", lambda m_: "X" if m_[0].isupper() else "x", t)\n        _func = lambda t: " ".join(w.lower() if w.lower() in _FW or not w.isalnum() else "CONTENT" for w in _re.findall(r"\\w+|[^\\w\\s]", t))\n        _P = pd.read_parquet(a.pool); _P = _P[_P.model.isin(LABELS)]\n        _fit = list(train.text.values) + list(_P.text.astype(str).values); _fy = np.r_[y, _P.model.map(L2I).values]\n        _A, _B = [], []\n        for view, lo, hi, mf, wv in [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]:\n            prep = _shape if view == "shape" else _func if view == "function" else (lambda s_: s_)\n            an = "char" if view == "shape" else "word" if view == "function" else view\n            o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)\n            if an == "word": o["token_pattern"] = r"(?u)\\b\\w+\\b|[^\\w\\s]"\n            _v = TfidfVectorizer(**o); _A.append(_v.fit_transform([prep(t) for t in _fit]) * wv); _B.append(_v.transform([prep(t) for t in test.text.values]) * wv)\n        _d = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight=\'balanced\').fit(_hs(_A, format=\'csr\'), _fy).decision_function(_hs(_B, format=\'csr\'))\n        LP[\'svc4\'] = (_d - _d.max(1, keepdims=True) - np.log(np.exp(_d - _d.max(1, keepdims=True)).sum(1, keepdims=True))) * a.svc_w\n        del _A, _B; gc.collect(); log(\'svc4 done\')\nif a.distort_w > 0:  # Stamatatos-style text distortion: keep top-100 frequent words + punctuation, mask the rest\n    import re, collections\n    wre = re.compile(r"[A-Za-z’\']+")\n    cnt = collections.Counter(w.lower() for t in FIT_T for w in wre.findall(t))\n    keep = {w for w, _ in cnt.most_common(100)}\n    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else \'*\', t) for t in arr]\n    Dtr, Dte = dist(FIT_T), dist(test.text.values)\n    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer=\'char\', ngram_range=(2, 6), min_df=2),\n          TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\\s]+", ngram_range=(1, 3), min_df=2)]\n    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr(); Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()\n    LP[\'distort\'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w\n    log(\'distort done\')\n\n# ---------------- 3. LM embeddings\nfrom transformers import AutoTokenizer, AutoModel\n@torch.no_grad()\ndef embed(spec, texts):\n    tok = AutoTokenizer.from_pretrained(spec[\'path\'])\n    if tok.pad_token is None: tok.pad_token = tok.eos_token\n    kw = dict(device_map=\'auto\')\n    q = spec.get(\'quant\', \'fp16\')\n    if q == \'4bit\':\n        from transformers import BitsAndBytesConfig\n        kw[\'quantization_config\'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_compute_dtype=torch.float16)\n    elif q == \'8bit\':\n        from transformers import BitsAndBytesConfig\n        kw[\'quantization_config\'] = BitsAndBytesConfig(load_in_8bit=True)\n    kw[\'dtype\'] = torch.bfloat16 if q == \'bf16\' else torch.float16\n    m = AutoModel.from_pretrained(spec[\'path\'], **kw).eval()\n    enc = tok(list(texts), truncation=True, max_length=512)[\'input_ids\']\n    order = np.argsort([len(x) for x in enc]); out = None; bs = spec.get(\'bs\', 16); nan_rows = 0\n    first = next(m.parameters()).device\n    for i in range(0, len(order), bs):\n        b = order[i:i + bs]; L = max(len(enc[j]) for j in b)\n        ids = torch.full((len(b), L), tok.pad_token_id); att = torch.zeros((len(b), L), dtype=torch.long)\n        for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1\n        hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True).hidden_states\n        nL = len(hs) - 1; sel = spec[\'layers_abs\'] if \'layers_abs\' in spec else [max(1, round(f * nL)) for f in spec[\'layers\']]\n        pooled = []\n        for l in sel:\n            h = hs[l].float(); w = att.to(h.device).unsqueeze(-1).float()\n            pooled.append(((h * w).sum(1) / w.sum(1)).cpu())\n        pooled = torch.cat(pooled, 1).numpy()\n        nan_rows += int((~np.isfinite(pooled)).any(1).sum())\n        if out is None: out = np.zeros((len(texts), pooled.shape[1]), np.float32); log(\'  layers\', sel, \'of\', nL)\n        out[b] = pooled\n    del m; gc.collect(); torch.cuda.empty_cache()\n    if nan_rows: log(\'  WARNING non-finite rows:\', nan_rows)\n    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)\n\nTPL = \'Identify which AI language model wrote the following passage, based only on its writing style.\\n\\nPassage:\\n{}\\n\\nThe author model is:\'\n@torch.no_grad()\ndef lora_predict(spec, texts):\n    """LoRA classifier WITHOUT peft: merge W += (B @ A) * alpha/r into the base weights, load the trained score head."""\n    from transformers import AutoModelForSequenceClassification\n    from safetensors.torch import load_file\n    tok = AutoTokenizer.from_pretrained(spec[\'path\'])\n    if tok.pad_token is None: tok.pad_token = tok.eos_token\n    m = AutoModelForSequenceClassification.from_pretrained(spec[\'path\'], num_labels=18, dtype=torch.float16, device_map=\'auto\')\n    m.config.pad_token_id = tok.pad_token_id\n    cfg = json.load(open(os.path.join(spec[\'adapter\'], \'adapter_config.json\')))\n    scale = cfg[\'lora_alpha\'] / cfg[\'r\']\n    sd = load_file(os.path.join(spec[\'adapter\'], \'adapter_model.safetensors\'))\n    mods = dict(m.named_modules()); n_merged = 0; head = None\n    for k, v in sd.items():\n        key = k.replace(\'base_model.model.\', \'\', 1)\n        if \'.lora_A.\' in key:\n            mod_name = key.split(\'.lora_A.\')[0]; B = sd[k.replace(\'.lora_A.\', \'.lora_B.\')]\n            lin = mods[mod_name]; W = lin.weight\n            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype); n_merged += 1\n        elif key.startswith(\'score.\') and \'weight\' in key and \'original_module\' not in key:\n            head = v\n    assert head is not None and n_merged > 0, (n_merged, [k for k in sd if \'score\' in k])\n    m.score.weight.copy_(head.to(m.score.weight.device, m.score.weight.dtype))\n    log(f\'  merged {n_merged} LoRA pairs (scale {scale}), head {tuple(head.shape)}\')\n    del mods, sd, lin, W, B, head; gc.collect()  # drop every reference to the model\'s modules\n    m.eval()\n    enc = tok(list(texts), truncation=True, max_length=512)[\'input_ids\']\n    order = np.argsort([len(x) for x in enc]); out = np.zeros((len(texts), 18), np.float32); bs = spec.get(\'bs\', 16)\n    first = next(m.parameters()).device\n    for i in range(0, len(order), bs):\n        b = order[i:i + bs]; L = max(len(enc[j]) for j in b)\n        ids = torch.full((len(b), L), tok.pad_token_id); att = torch.zeros((len(b), L), dtype=torch.long)\n        for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1\n        lo = m(input_ids=ids.to(first), attention_mask=att.to(first)).logits.float()\n        out[b] = torch.log_softmax(lo, -1).cpu().numpy()\n    del m; gc.collect(); torch.cuda.empty_cache()\n    return out\n\n@torch.no_grad()\ndef l24_predict(spec, texts, groups=None, train_texts=None):\n    """LoRA24-recipe member(s): truncated base loaded ONCE; for each adapter dir in spec[\'adapters\'] (or spec[\'adapter\'])\n    merge W += B A * s, compute masked mean-pooled multi-layer features, read plain (+ centred) heads, then unmerge.\n    Returns log-probs averaged over adapters (plain, centred-or-None)."""\n    from transformers import AutoConfig\n    from safetensors.torch import load_file\n    adapters = spec.get(\'adapters\') or [spec[\'adapter\']]\n    tok = AutoTokenizer.from_pretrained(spec[\'path\'])\n    if tok.pad_token is None: tok.pad_token = tok.eos_token\n    cfg = AutoConfig.from_pretrained(spec[\'path\']); tc = cfg.text_config if hasattr(cfg, \'text_config\') else cfg\n    tc.num_hidden_layers = spec[\'keep\']; tc.use_cache = False\n    if getattr(tc, \'layer_types\', None) is not None: tc.layer_types = tc.layer_types[:spec[\'keep\']]\n    m = AutoModel.from_pretrained(spec[\'path\'], config=cfg, dtype=torch.float16, device_map=\'auto\').eval()\n    if spec.get(\'no_final_norm\'): m.norm = torch.nn.Identity()\n    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec[\'layers\']\n    _eos = spec.get(\'eos_id\')   # checkpoints trained with an end token appended (Nishant\'s) need the same id at inference\n    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get(\'max_len\', 384) - (1 if _eos is not None else 0))[\'input_ids\']]\n    enc = _tk(texts)\n    order = np.argsort([len(x) for x in enc]); bs = spec.get(\'bs\', 16)\n    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))\n    P_acc, C_acc = 0.0, 0.0; FEATS = None; OVF = [0]\n    if train_texts is not None:\n        enc_tr = _tk(train_texts)\n    def featurize(enc_, norms, hid):\n        order_ = np.argsort([len(x) for x in enc_]); Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)\n        for i in range(0, len(order_), bs):\n            b = order_[i:i + bs]; Lm = max(len(enc_[j]) for j in b)\n            ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)\n            for r, j in enumerate(b): ids[r, :len(enc_[j])] = torch.tensor(enc_[j]); att[r, :len(enc_[j])] = 1\n            hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states\n            if not all(torch.isfinite(hs[l]).all() for l in L):\n                m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1\n            mask = att.unsqueeze(-1).float(); mask[:, 0] = 0\n            Z_[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()\n        return Z_\n    for ai, ad in enumerate(adapters):\n        adp = os.path.join(ad, \'adapter\') if os.path.exists(os.path.join(ad, \'adapter\', \'adapter_config.json\')) else ad\n        acfg = json.load(open(os.path.join(adp, \'adapter_config.json\'))); scale = acfg[\'lora_alpha\'] / acfg[\'r\']\n        sd = load_file(os.path.join(adp, \'adapter_model.safetensors\')); deltas = []\n        for k, v in sd.items():\n            if \'.lora_A.\' not in k: continue\n            name = k.replace(\'base_model.model.\', \'\', 1).split(\'.lora_A.\')[0]\n            if name not in mods and (\'model.\' + name) in mods: name = \'model.\' + name\n            W = mods[name].weight; B = sd[k.replace(\'.lora_A.\', \'.lora_B.\')]\n            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)\n            deltas.append((W, B, v))   # keep only the low-rank factors; the dense delta is recomputed to unmerge\n        if os.path.exists(os.path.join(ad, \'heads.pt\')): H = torch.load(os.path.join(ad, \'heads.pt\'), map_location=\'cpu\')\n        else: H = dict(load_file(os.path.join(ad, \'heads.safetensors\')))\n        if os.path.exists(os.path.join(ad, \'meta.json\')):   # remap head rows from the checkpoint\'s label order to ours\n            _lab = json.load(open(os.path.join(ad, \'meta.json\')))[\'labels\']; _perm = [_lab.index(l) for l in LABELS]\n            for _p in (\'plain\', \'centered\'): H[f\'{_p}.weight\'] = H[f\'{_p}.weight\'][_perm]; H[f\'{_p}.bias\'] = H[f\'{_p}.bias\'][_perm]\n        hid = H[\'norms.0.weight\'].shape[0]\n        norms = [torch.nn.LayerNorm(hid) for _ in L]\n        for i, nm in enumerate(norms): nm.load_state_dict({\'weight\': H[f\'norms.{i}.weight\'], \'bias\': H[f\'norms.{i}.bias\']})\n        Z = np.zeros((len(texts), hid * len(L)), np.float32)\n        for i in range(0, len(order), bs):\n            b = order[i:i + bs]; Lm = max(len(enc[j]) for j in b)\n            ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)\n            for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1\n            hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states\n            if not all(torch.isfinite(hs[l]).all() for l in L):   # fp16 overflow (un-normed deep states): redo this batch in bf16\n                m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1\n            mask = att.unsqueeze(-1).float(); mask[:, 0] = 0\n            Z[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()\n        if train_texts is not None and ai == len(adapters) - 1:   # full-data adapter: features for the Gaussian head\n            FEATS = (Z.copy(), featurize(enc_tr, norms, hid))\n        for W, B, A_ in deltas:   # unmerge before the next adapter\n            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)\n        torch.cuda.empty_cache()\n        del deltas, sd; gc.collect()\n        hd = lambda p, X: lsm(X @ H[f\'{p}.weight\'].float().numpy().T + H[f\'{p}.bias\'].float().numpy())\n        P_acc = P_acc + hd(\'plain\', Z) / len(adapters)\n        if groups is not None:\n            Zc = Z.copy()\n            for g in np.unique(groups):\n                ii = groups == g; Zc[ii] -= Zc[ii].mean(0)\n            C_acc = C_acc + hd(\'centered\', Zc) / len(adapters)\n        log(f\'  l24 {spec["name"]}: adapter {os.path.basename(ad)} done (bf16 redo batches {OVF[0]})\')\n    del m, mods; gc.collect(); torch.cuda.empty_cache()\n    if train_texts is not None: return P_acc, (C_acc if groups is not None else None), FEATS\n    return P_acc, (C_acc if groups is not None else None)\n\ndef gpu_mem():\n    return \' \'.join(f\'gpu{i}:{torch.cuda.memory_allocated(i)/1e9:.1f}GB\' for i in range(torch.cuda.device_count()))\n\nspecs = json.loads(a.models) if a.models else []\nall_text = np.concatenate([train.text.values, test.text.values])\nfor spec in specs:\n    if spec.get(\'type\') == \'l24\':\n        log(\'l24\', spec[\'name\'])\n        if len(test) < 50:\n            _p40, _ = l24_predict(spec, train.text.values[:40]); _acc = float((_p40.argmax(1) == y[:40]).mean())\n            log(f\'  {spec["name"]} in-sample check acc {_acc:.2f}\'); assert _acc > 0.5, f\'{spec["name"]} l24 does not reproduce its fit\'\n        if spec.get(\'gauss\', 0) > 0:\n            plain, cent, (Zte_g, Ztr_g) = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None, train_texts=train.text.values)\n            import gauss as _ga\n            prm = _ga.fit(Ztr_g, y, Gtr, dict(red=\'wpca\', d=128, se_shrink=\'lw\', sp_shrink=\'lw\', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))\n            prm[\'T1\'] = prm[\'Tg\'] = 3.0\n            _sz = np.bincount(Gte)[Gte]; _rows = (_sz >= 2) & (_sz <= 18)\n            _q = _ga.scores(Ztr_g[[0, 1]], np.zeros(2, dtype=int), prm); assert np.allclose(np.exp(_q).sum(1), 1)\n            _G = np.zeros((len(test), 18))\n            if _rows.any():\n                _G[_rows] = _ga.scores(Zte_g[_rows], np.unique(Gte[_rows], return_inverse=True)[1], prm)\n            LP[spec[\'name\'] + \'_gauss\'] = _G * spec[\'gauss\']; log(f"  gaussian head {spec[\'name\']}: {int(_rows.sum())} grouped test rows, weight {spec[\'gauss\']}")\n        else:\n            plain, cent = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None)\n        LP[spec[\'name\']] = plain * spec.get(\'w\', 1.0)\n        if cent is not None:\n            rows = np.bincount(Gte)[Gte] >= 2\n            LP[spec[\'name\'] + \'_twin\'] = np.where(rows[:, None], cent, 0.0) * a.adapt_w * spec.get(\'w\', 1.0) * spec.get(\'twin\', 1.0)\n        gc.collect(); torch.cuda.empty_cache(); log(spec[\'name\'], \'done\', gpu_mem()); continue\n    if spec.get(\'type\') == \'enc\':   # fully fine-tuned encoder (DeBERTa-style): fp32, mean of the last layer, linear head\n        log(\'enc\', spec[\'name\']); from safetensors.torch import load_file as _lf\n        _ck = spec[\'ckpt\']; _meta = json.load(open(os.path.join(_ck, \'meta.json\')))\n        _perm = [_meta[\'labels\'].index(l) for l in LABELS]\n        _hd = _lf(os.path.join(_ck, \'head.safetensors\')); _W = _hd[\'weight\'].float().numpy()[_perm]; _b = _hd[\'bias\'].float().numpy()[_perm]\n        _etok = AutoTokenizer.from_pretrained(os.path.join(_ck, \'encoder\'))\n        _enc = AutoModel.from_pretrained(os.path.join(_ck, \'encoder\'), dtype=torch.float32).to(\'cuda:0\').eval()\n        def _enc_feats(texts):\n            _ids = _etok(list(texts), truncation=True, max_length=_meta[\'max_len\'])[\'input_ids\']\n            _ord = np.argsort([len(x) for x in _ids], kind=\'stable\'); Z_ = np.zeros((len(_ids), _enc.config.hidden_size), np.float32)\n            with torch.inference_mode():\n                for _s in range(0, len(_ord), 32):\n                    _idx = _ord[_s:_s + 32]; _bt = _etok.pad({\'input_ids\': [_ids[i] for i in _idx]}, return_tensors=\'pt\').to(\'cuda:0\')\n                    _m = _bt[\'attention_mask\'].unsqueeze(-1).float()\n                    Z_[_idx] = ((_enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()\n            return Z_\n        if len(test) < 50:\n            _acc = float(((_enc_feats(train.text.values[:40]) @ _W.T + _b).argmax(1) == y[:40]).mean())\n            log(f\'  {spec["name"]} in-sample check acc {_acc:.2f}\'); assert _acc > 0.5, spec[\'name\']\n        _zz = _enc_feats(test.text.values) @ _W.T + _b\n        LP[spec[\'name\']] = (_zz - _zz.max(1, keepdims=True) - np.log(np.exp(_zz - _zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get(\'w\', 1.0)\n        del _enc; gc.collect(); torch.cuda.empty_cache(); log(spec[\'name\'], \'done\', gpu_mem()); continue\n    if spec.get(\'type\') == \'head\':\n        log(\'head\', spec[\'name\']); H = np.load(spec[\'head\'])\n        Ete = embed(dict(spec, layers=sorted(float(v) for v in H[\'layers\'])), test.text.values)  # extraction stored layers in sorted order\n        if len(test) < 50:  # interactive run: in-sample sanity check on 40 train rows (head was fit on them)\n            _E40 = embed(dict(spec, layers=sorted(float(v) for v in H[\'layers\'])), train.text.values[:40])\n            _acc = float(((((_E40 - H[\'mu\']) / H[\'sd\']) @ H[\'W\'] + H[\'b\']).argmax(1) == y[:40]).mean())\n            log(f\'  {spec["name"]} in-sample check acc {_acc:.2f}\'); assert _acc > 0.5, f\'{spec["name"]} head does not reproduce its fit\'\n        Z = (Ete - H[\'mu\']) / H[\'sd\']; zz = Z @ H[\'W\'] + H[\'b\']\n        LP[spec[\'name\']] = (zz - zz.max(1, keepdims=True) - np.log(np.exp(zz - zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get(\'w\', 1.0)\n        gc.collect(); torch.cuda.empty_cache(); log(spec[\'name\'], \'done\', gpu_mem()); continue\n    if spec.get(\'type\') == \'lora\':\n        log(\'lora\', spec[\'name\']); LP[spec[\'name\']] = lora_predict(spec, test.text.values) * spec.get(\'w\', 1.0)\n        gc.collect(); torch.cuda.empty_cache(); log(spec[\'name\'], \'done\', gpu_mem()); continue\n    name = spec[\'name\']; cp = os.path.join(a.cache, f"{name}_{\'sim\' + a.sim + a.sim_k if a.sim else \'real\'}.npy")\n    if os.path.exists(cp): E = np.load(cp)\n    else:\n        log(\'embedding\', name); E = embed(spec, all_text); np.save(cp, E)\n    Etr, Ete = E[:len(train)], E[len(train):]\n    r = torch_lr(Etr, y, Ete, spec.get(\'C\', a.C))\n    if a.perpassage:\n        LP[name] = r * spec.get(\'w\', 1.0)\n        if a.adapt_w > 0:\n            sizes = np.bincount(Gte)[Gte]; rows = sizes >= 2\n            c = torch_lr(center_groups(Etr, Gtr), y, center_groups(Ete, Gte), spec.get(\'C\', a.C))\n            LP[name + \'_twin\'] = np.where(rows[:, None], c, 0.0) * a.adapt_w * spec.get(\'w\', 1.0)\n    else:\n        c = torch_lr(center_groups(Etr, Gtr), y, center_groups(Ete, Gte), a.C)\n        LP[name] = (r + c) * spec.get(\'w\', 1.0)\n    gc.collect(); torch.cuda.empty_cache(); log(name, \'done\', gpu_mem())\n\n# ---------------- 4. combine + assign\n# typography guards: (mark, models that ever use it in train). Zero violations on the 38k matched pool for every rule.\nGUARDS = [(\' \\u2014 \', {\'claude-opus-4-6\', \'claude-opus-5\', \'glm-5.3\', \'glm-5.3-flash\'}),\n          (\'\\u2019\', {l for l in LABELS if not l.startswith((\'claude\', \'glm\'))}),\n          (\'"\', {\'claude-fable-5\', \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\', \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\', \'glm-5.3\', \'glm-5.3-flash\', \'gpt-5.5\', \'kimi-k3\'}),\n          (\'\\u201c\', {\'gemini-3.8-flash\', \'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\', \'kimi-k3\', \'qwen3.8-max\'}),\n          (\'\\u2013\', {\'claude-opus-5\', \'gemini-3.1-pro-preview\', \'glm-5.3-flash\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'kimi-k3\'})]\ntot = sum(LP.values())\ndef atlas_scores(bundle, rows, budget):\n    """Source-conditioned likelihood probabilities (public ATLAS bundle format) for test rows `rows`, in our label order.\n    Rows are scored shortest-first; raises if the token-proportional projection would overrun `budget` (s since start)."""\n    sys.path.insert(0, bundle)\n    for _m in [m_ for m_ in list(sys.modules) if m_.startswith((\'ntl_atlas\', \'ntl_rlcd\'))]: del sys.modules[_m]\n    from ntl_atlas.model import load_model as _atlas_load\n    from ntl_rlcd.data import LABELS as _alab\n    from safetensors.torch import load_file as _lf\n    sys.path.remove(bundle)\n    _cfg = json.load(open(os.path.join(bundle, \'atlas_bundle.json\')))[\'training_config\']\n    _am, _atok = _atlas_load(a.atlas_base, _alab, _cfg, device=\'auto\')\n    try:\n        _am.load_trainable(_lf(os.path.join(bundle, \'adapters.safetensors\'))); _am.eval()\n        _enc = _atok(test.text.values[rows].tolist(), truncation=True, max_length=_cfg[\'max_length\'])[\'input_ids\']\n        _ord = np.argsort([len(t) for t in _enc], kind=\'stable\'); out = np.zeros((len(rows), 18), np.float32); _t1 = time.time()\n        _ctok = np.cumsum(np.array([len(_enc[int(i)]) for i in _ord], np.float64))\n        for _b in range(0, len(_ord), a.atlas_bs):\n            _idx = _ord[_b:_b + a.atlas_bs]\n            _pk = _atok.pad({\'input_ids\': [_enc[int(i)] for i in _idx]}, padding=True, return_tensors=\'pt\').to(_am.device)\n            with torch.inference_mode(): _p, _ = _am.score(_pk[\'input_ids\'], _pk[\'attention_mask\'], source_chunk=a.atlas_chunk)\n            out[_idx] = _p.float().cpu().numpy(); _done = _b + len(_idx)\n            if _b % (25 * a.atlas_bs) == 0:\n                _proj = (time.time() - T0) + (time.time() - _t1) / _ctok[_done - 1] * (_ctok[-1] - _ctok[_done - 1]) * 1.1   # token-proportional\n                log(f\'atlas {os.path.basename(bundle)} {_done}/{len(_ord)} projected finish {_proj:.0f}s\')\n                if _done > 50 and _proj > budget: raise RuntimeError(\'atlas would overrun the time budget\')\n                if a.atlas_probe and _done >= a.atlas_probe: raise RuntimeError(f\'atlas probe stop; projected {_proj:.0f}s\')\n        assert np.isfinite(out).all() and np.allclose(out.sum(1), 1, atol=1e-4)\n        return out[:, [_alab.index(l) for l in LABELS]]\n    finally:\n        del _am; gc.collect(); torch.cuda.empty_cache()\n\nCOMP_P = None\nif a.comp_w > 0 and a.comp_path and os.path.exists(a.comp_path):   # complementary public stack, run in its own process\n    try:\n        _cm = json.load(open(a.comp_path.replace(\'comp_logit.npy\', \'comp_meta.json\')))\n        assert _cm[\'ids\'] == test[\'id\'].astype(str).tolist(), \'comp stack row order differs\'\n        _cl = np.load(a.comp_path)[:, [_cm[\'labels\'].index(l) for l in LABELS]]\n        COMP_P = np.exp(_cl - _cl.max(1, keepdims=True)); COMP_P /= COMP_P.sum(1, keepdims=True)\n        assert np.isfinite(COMP_P).all(); log(f"comp stack loaded ({_cm[\'seconds\']:.0f}s)")\n    except Exception as _e:\n        log(\'COMP SKIPPED:\', repr(_e)); COMP_P = None\nelif a.comp_w > 0: log(\'COMP SKIPPED: no stack output\')\n_bundles = [d for d in a.atlas_dir.split(\',\') if d]; _aw = [float(x) for x in str(a.atlas_w).split(\',\')] if _bundles else []\nif _bundles and len(_aw) == 1: _aw = _aw * len(_bundles)\nif (a.sinkhorn > 0 and len(test) >= 18) or _bundles or COMP_P is not None:\n    wsum = sum(abs(v).max() > 0 for v in LP.values())\n    lpn = tot / max(1, wsum) if a.sk_div else tot; P = np.exp(lpn - lpn.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)\n    if COMP_P is not None: P = (1 - a.comp_w) * P + a.comp_w * COMP_P; P /= P.sum(1, keepdims=True)\n    if _bundles and sum(_aw) > 0:\n        _s = np.sort(P, 1); _margin = _s[:, -1] - _s[:, -2]\n        rows = np.where(_margin < a.atlas_margin)[0] if a.atlas_margin > 0 and len(test) >= 50 else np.arange(len(test))\n        log(f\'atlas on {len(rows)}/{len(test)} rows (top-2 margin < {a.atlas_margin})\')\n        mixed = P.copy(); _wleft = 1.0\n        for _bd, _w in zip(_bundles, _aw):\n            try:\n                A_ = atlas_scores(_bd, rows, a.atlas_budget)\n                mixed[rows] = mixed[rows] - _w * P[rows] + _w * A_; log(f\'atlas {_bd} mixed at {_w}\', gpu_mem())\n            except Exception as _e:\n                log(\'ATLAS SKIPPED:\', _bd, repr(_e))\n        P = mixed / mixed.sum(1, keepdims=True)\n    if os.environ.get(\'MERGE26_FUSION_PATHS\'):\n        from fusion import add_to_nlm\n        P = add_to_nlm(test.id.astype(str).tolist(), LABELS, P,\n            json.loads(os.environ[\'MERGE26_FUSION_PATHS\']))\n        np.savez_compressed(os.path.join(a.cache,\'merge26_prebalance.npz\'),\n            ids=test.id.astype(str).to_numpy(),labels=np.asarray(LABELS),probabilities=P)\n        log(\'MERGE26_FUSED_7225_1275_15_NO27B\')\n    Q = P.copy()\n    if not (a.sinkhorn > 0 and len(test) >= 18): tot = np.log(P + 1e-12)\n    if a.sinkhorn > 0 and len(test) >= 18:\n        for _ in range(100):\n            Q *= (len(Q) / 18) / Q.sum(0, keepdims=True); Q /= Q.sum(1, keepdims=True)\n        tot = (1 - a.sinkhorn) * np.log(P + 1e-12) + a.sinkhorn * np.log(Q + 1e-12)\nif a.post and a.perpassage:\n    _nfix = 0\n    for _mark, _allowed in GUARDS:\n        _forb = np.array([l not in _allowed for l in LABELS])\n        for _i, _t in enumerate(test.text.values):\n            if _mark in _t and LABELS[int(tot[_i].argmax())] not in _allowed:\n                tot[_i, _forb] = -1e9; _nfix += 1\n    _sz = np.bincount(Gte)[Gte]; _nh = 0\n    for _g in np.unique(Gte[(_sz >= 12) & (_sz <= 18)]):\n        _ii = np.where(Gte == _g)[0]; _r, _c = linear_sum_assignment(-tot[_ii]); _nh += int((tot[_ii].argmax(1)[_r] != _c).sum())\n        tot[_ii[_r]] = -1e9; tot[_ii[_r], _c] = 0.0\n    log(f\'post: em-dash guard fixed {_nfix}, dense-cluster Hungarian changed {_nh}\')\npred = tot.argmax(1) if a.perpassage else hungarian(tot, Gte)\nif a.sim:\n    from sklearn.metrics import balanced_accuracy_score as bas\n    yt = test_truth.model.map(L2I).values\n    for k, v_ in LP.items(): log(f\'  {k:12s} arg {bas(yt, v_.argmax(1)):.4f} hung {bas(yt, hungarian(v_, Gte)):.4f}\')\n    log(f\'ENSEMBLE arg {bas(yt, tot.argmax(1)):.4f} | hung(est clusters) {bas(yt, pred):.4f} | hung(true prompts) {bas(yt, hungarian(tot, test_truth.prompt_id.values)):.4f}\')\n    np.save(os.path.join(a.cache, f\'sim{a.sim}_lp.npy\'), tot)\nif a.save_lp:\n    wtot = sum(abs(float(s_.get(\'w\', 1.0))) for s_ in specs) + (a.tfidf_w if a.tfidf_w > 0 else 0) + (a.distort_w if a.distort_w > 0 else 0) + (a.svc_w if a.svc_w > 0 else 0)\n    from scipy.special import log_softmax as _lsm\n    np.save(a.save_lp, _lsm(tot / max(wtot, 1e-6), axis=1).astype(np.float32)); log(\'saved\', a.save_lp, \'norm\', wtot)\nsub = pd.DataFrame({\'id\': test.id.values, \'model\': [LABELS[i] for i in pred]})\nassert sub.model.isin(LABELS).all() and sub.id.is_unique and len(sub) == len(test)\nsub.to_parquet(a.out, index=False); log(\'wrote\', a.out, sub.model.value_counts().to_dict())\n')
    (root/'input_stage.py').write_text("import shutil,tempfile,time,json\nfrom pathlib import Path\nfrom contextlib import contextmanager\n@contextmanager\ndef staged_dataset_tree(source, files, *, label, input_root='/kaggle/input', temp_root='/tmp'):\n    source = Path(source)\n    input_root = Path(input_root)\n    temp_root = Path(temp_root)\n    assert source.is_dir() and source.resolve().is_relative_to(input_root.resolve())\n    assert label and label.replace('-', '').isalnum()\n    assert files and all('/' not in name and '..' not in name and size > 0\n                         for name, size in files.items())\n    total = sum(files.values())\n    for name, size in files.items():\n        item = source / name\n        assert item.is_file() and item.stat().st_size == size, (label, name, size)\n    free = shutil.disk_usage(temp_root).free\n    assert free > total + 2 * 2**30, ('Insufficient /tmp disk for explicit staging', label, total, free)\n    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))\n    assert destination.resolve().is_relative_to(temp_root.resolve())\n    start = time.monotonic()\n    try:\n        for name, size in files.items():\n            target = destination / name\n            with (source / name).open('rb') as reader, target.open('wb') as writer:\n                while block := reader.read(16 * 1024 * 1024):\n                    writer.write(block)\n            assert target.stat().st_size == size\n            print(json.dumps({'stage': label, 'file': name, 'bytes': size,\n                              'elapsed_seconds': round(time.monotonic() - start, 2)}), flush=True)\n        yield destination\n    finally:\n        assert destination.resolve().is_relative_to(temp_root.resolve())\n        shutil.rmtree(destination)\n        print(json.dumps({'stage': label, 'event': 'STAGING_CLEANED'}), flush=True)\n")
    (root/'nf4_linear_common.py').write_text("# Packed linear and host embedding credited to lightningv08.\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom bitsandbytes.functional import dequantize_4bit\nclass HostEmbedding(nn.Module):\n    def __init__(self, weight):\n        super().__init__(); self.weight_cpu = weight\n    def forward(self, ids):\n        return F.embedding(ids.cpu(), self.weight_cpu).to(ids.device)\n\nclass NF4InferenceLinear(nn.Module):\n    def __init__(self, packed, state, shape, rank, device, resident):\n        super().__init__()\n        self.packed = packed.to(device) if resident else packed\n        self.state, self.compute_device = state, device\n        self.in_features, self.out_features = shape\n        if rank:\n            self.lora_A = nn.Parameter(torch.empty(rank, shape[0], device=device, dtype=torch.float32), requires_grad=False)\n            self.lora_B = nn.Parameter(torch.empty(shape[1], rank, device=device, dtype=torch.float32), requires_grad=False)\n        else:\n            self.register_parameter('lora_A', None); self.register_parameter('lora_B', None)\n    def forward(self, value):\n        original_device = value.device\n        value = value.to(self.compute_device)\n        weight = dequantize_4bit(self.packed.to(self.compute_device), self.state).to(value.dtype)\n        base = F.linear(value, weight)\n        del weight\n        if self.lora_A is not None:\n            with torch.autocast('cuda', dtype=value.dtype):\n                extra = F.linear(F.linear(value, self.lora_A), self.lora_B)\n            base = base + 2. * extra.to(base.dtype)\n        return base.to(original_device)\n")
    (root/'night24_cpu_worker.py').write_text('"""CPU-only published spectral readouts, after the GPU complement has exited."""\nimport os\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'\'\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'2\')\nos.environ.setdefault(\'MKL_NUM_THREADS\', \'2\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\', \'2\')\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport sys\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom readouts import predict_bank, LAYERS\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    for name in [\'features\', \'feature-metadata\', \'query\', \'bank\', \'bank-metadata\', \'out\']:\n        parser.add_argument(\'--\' + name, required=True)\n    args = parser.parse_args()\n    raw_path = Path(args.features)\n    z = None\n    try:\n        raw = json.loads(Path(args.feature_metadata).read_text(encoding=\'utf-8\'))\n        test = pd.read_csv(args.query, dtype={\'id\': str})\n        texts = test.text.fillna(\'\').astype(str).tolist()\n        assert raw[\'ids\'] == test.id.tolist() and len(set(raw[\'ids\'])) == len(test)\n        bank_audit = json.loads(Path(args.bank_metadata).read_text(encoding=\'utf-8\'))\n        assert bank_audit[\'full_fit\'] and not bank_audit[\'fold_adapter\'] and not bank_audit[\'adapter_training\']\n        assert not bank_audit[\'raw_synthetic_data_in_export\']\n        record = bank_audit[\'banks\'][\'synthetic4404\']\n        checkpoint_audit = json.loads(Path(raw[\'checkpoint_metadata\']).read_text(encoding=\'utf-8\'))\n        bank = joblib.load(args.bank)\n        # Original compatibility checks; never scan large model weight files here.\n        assert bank[\'adapter_sha256\'] == record[\'adapter_sha256\'] == checkpoint_audit[\'files\'][\'adapter/adapter_model.safetensors\']\n        assert checkpoint_audit[\'files\'][\'heads.safetensors\'] == \'2176ade29e6eb97bd3a6c204f944959f9757a80c6f1ff475898c23bce60ecd96\'\n        assert bank[\'adapter_sha256\'] == \'a6aff409ae2df897717875373d038635fe5012fbf93557c90aea3457fda0e7f8\'\n        assert bank[\'labels\'] == record[\'labels\'] == raw[\'labels\'] and bank[\'layers\'] == record[\'layers\'] == raw[\'layers\'] == LAYERS\n        assert bank[\'official_rows\'] == record[\'official_rows\'] == 2699\n        assert bank[\'synthetic_rows\'] == record[\'synthetic_rows\'] == 4404\n        z = np.load(raw_path, allow_pickle=False, mmap_mode=\'r\')\n        assert list(z.shape) == raw[\'shape\'] == [len(texts), 32768] and str(z.dtype) == raw[\'dtype\']\n        hp, hm = predict_bank(bank, z, texts)\n        np.savez_compressed(args.out, ids=test.id.to_numpy(dtype=str),\n                            labels=np.asarray(raw[\'labels\'], dtype=str), probabilities=hp, **hm)\n        print(\'NIGHT24_READOUTS_REAL_TEST_COMPLETE\', len(z), bank[\'synthetic_rows\'], flush=True)\n        assert \'torch\' not in sys.modules and \'transformers\' not in sys.modules\n    finally:\n        if z is not None:\n            del z\n            gc.collect()\n        raw_path.unlink(missing_ok=True)\n        Path(args.feature_metadata).unlink(missing_ok=True)\n        print(\'NIGHT24_RAW_FEATURES_CLEANED\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
    (root/'p3_train_full.py').write_text('"""Private fresh32 development folds; fixed recipe, no competition submission."""\nimport argparse\nimport json\nimport shutil\nimport time\nimport traceback\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom safetensors.torch import save_file\nfrom torch.nn import functional as F\n\nfrom p3_training_utils import enable_training, forward_logits, wanted_weights, parameter_sample, synchronize, sha\n\n\nclass BudgetPause(Exception):\n    pass\n\n\ndef bucket_batches(indices, lengths, seed, training=True, microbatch=4):\n    order=sorted(map(int,indices),key=lambda i:(int(lengths[i]),i))\n    rng=np.random.default_rng(seed)\n    if training:\n        windows=[order[i:i+64] for i in range(0,len(order),64)]\n        for window in windows:rng.shuffle(window)\n        order=[i for window in windows for i in window]\n    batches=[order[i:i+microbatch] for i in range(0,len(order),microbatch)]\n    if training:\n        partial=batches.pop() if len(batches[-1])<microbatch else None\n        rng.shuffle(batches)\n        if partial is not None:batches.append(partial)\n    assert sorted(i for batch in batches for i in batch)==sorted(map(int,indices))\n    return batches\n\n\ndef accumulation_groups(batches, target=16):\n    pending=[];count=0\n    for batch in batches:\n        assert 0<len(batch)<=8\n        if count+len(batch)>target:\n            yield pending\n            pending=[];count=0\n        pending.append(batch);count+=len(batch)\n        if count==target:\n            yield pending\n            pending=[];count=0\n    if pending:yield pending\n\n\ndef optimizer_update(model, params, optimizer, scaler, batches, tokens, targets, pack):\n    optimizer.zero_grad(set_to_none=True)\n    count=sum(map(len,batches));loss_sum=0.\n    for rows in batches:\n        scores=forward_logits(model,pack(tokens,rows))\n        loss=F.cross_entropy(scores,torch.tensor(targets[rows],device=model.device),reduction=\'sum\')/count\n        assert torch.isfinite(loss)\n        scaler.scale(loss).backward();loss_sum+=float(loss.detach())\n        del scores,loss\n    for name,param in params.items():assert param.grad is not None and torch.isfinite(param.grad).all(),name\n    scaler.unscale_(optimizer)\n    norm=torch.nn.utils.clip_grad_norm_(list(params.values()),1.,error_if_nonfinite=True)\n    scaler.step(optimizer);scaler.update()\n    return count,loss_sum,float(norm)\n\n\ndef restore_optimizer(optimizer, scaler, state, *, phase, indices, seed, epochs,\n                      lora_lr, head_lr, names, protocol_sha, adapter_sha):\n    """Fail closed on mismatched recipe/cursor/state before any resumed update."""\n    expected=dict(phase=phase,indices=list(map(int,indices)),seed=seed,epochs=epochs,\n                  lora_lr=lora_lr,head_lr=head_lr,trainable_names=list(names),\n                  protocol_sha256=protocol_sha,adapter_sha256=adapter_sha)\n    for key,value in expected.items():assert state[key]==value,(\'Resume mismatch\',key)\n    assert 0<=state[\'epoch_zero_based\']<epochs and state[\'next_group\']>=0\n    optimizer.load_state_dict(state[\'optimizer\']);scaler.load_state_dict(state[\'scaler\'])\n    torch.set_rng_state(state[\'cpu_rng\'])\n    if torch.cuda.is_available():torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n\n\ndef run(args):\n    from qwen32_nf4 import Qwen32Classifier, REVISION, EOS\n    began=time.monotonic();out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    protocol=json.loads(Path(args.protocol).read_text());protocol_sha=sha(args.protocol)\n    previous=Path(args.previous_output) if args.previous_output else None\n    progress=dict(status=\'P3_ACTUAL_FRESH32_RUNNING\',full64=True,private=True,RAW_exported=False,\n        competition_submission=False,protocol_sha256=protocol_sha,completed=[],steps=0,\n        folds_requested=args.folds,source_revision=REVISION,initial_fresh_seed=protocol[\'seed\'])\n    resume=None;model=None;params=None\n\n    def write():\n        progress[\'wall_seconds\']=time.monotonic()-began\n        temp=out/\'progress.tmp\';temp.write_text(json.dumps(progress,indent=2));temp.replace(out/\'progress.json\')\n\n    def save_weights(stem,details):\n        path=out/(stem+\'.safetensors\');temp=path.with_suffix(\'.safetensors.tmp\')\n        save_file({n:p.detach().cpu().contiguous().clone() for n,p in wanted_weights(model).items()},str(temp));temp.replace(path)\n        metadata=dict(model.metadata);metadata.update(details,benchmark_only=False,not_for_scoring=True,\n            RAW_exported=False,protocol_sha256=protocol_sha,full64=True)\n        path.with_suffix(\'.json\').write_text(json.dumps(metadata,indent=2));return path\n\n    def check_adapter(path,phase):\n        meta=json.loads(path.with_suffix(\'.json\').read_text())\n        assert meta[\'protocol_sha256\']==protocol_sha and meta[\'stage\']==phase\n        assert meta[\'benchmark_only\'] is False and meta[\'not_for_scoring\'] is True\n        assert meta[\'source_revision\']==REVISION and meta[\'classification_token_id\']==EOS\n        return meta\n\n    def preserve(path):\n        # Continuation outputs retain the already accepted common stage, so a\n        # later continuation never has to recompute it from a fresh adapter.\n        target=out/path.name\n        if path.resolve()!=target.resolve():\n            shutil.copyfile(path,target)\n            shutil.copyfile(path.with_suffix(\'.json\'),target.with_suffix(\'.json\'))\n            assert sha(path)==sha(target)\n        return target\n\n    def pack(tokens,rows):\n        return model.tokenizer.pad({\'input_ids\':[tokens[i]+[EOS] for i in rows]},\n            padding=True,pad_to_multiple_of=64,return_tensors=\'pt\').to(model.device)\n\n    def train(phase,tokens,targets,indices,epochs,lora_lr,head_lr,seed):\n        nonlocal resume\n        model.model.train();model.norms.train();model.plain.train()\n        groups=[dict(params=[p for n,p in params.items() if n.startswith(\'backbone.\')],lr=lora_lr),\n                dict(params=[p for n,p in params.items() if not n.startswith(\'backbone.\')],lr=head_lr)]\n        optimizer=torch.optim.AdamW(groups,betas=(.9,.999),eps=1e-8,weight_decay=0)\n        assert {id(p) for g in optimizer.param_groups for p in g[\'params\']}=={id(p) for p in params.values()}\n        scaler=torch.amp.GradScaler(\'cuda\',init_scale=128,growth_interval=10000)\n        lengths=np.asarray([len(t)+1 for t in tokens]);local_step=stage_rows=0;epoch_start=group_start=0\n        stage_started=time.monotonic()\n        if resume is not None and resume[\'phase\']==phase:\n            path=previous/\'RESUME_ONLY.safetensors\';check_adapter(path,\'p3_resume_only\');model.reload(path)\n            restore_optimizer(optimizer,scaler,resume,phase=phase,indices=indices,seed=seed,epochs=epochs,\n                lora_lr=lora_lr,head_lr=head_lr,names=params,protocol_sha=protocol_sha,adapter_sha=sha(path))\n            epoch_start=resume[\'epoch_zero_based\'];group_start=resume[\'next_group\']\n            local_step=resume[\'local_step\'];stage_rows=resume[\'stage_rows\'];progress[\'steps\']=resume[\'total_steps\']\n            progress[\'resumed_exact_cursor\']=dict(phase=phase,epoch=epoch_start,next_group=group_start)\n            resume=None\n        for epoch in range(epoch_start,epochs):\n            batches=list(accumulation_groups(bucket_batches(indices,lengths,seed+epoch)))\n            start_group=group_start if epoch==epoch_start else 0\n            assert 0<=start_group<=len(batches)\n            for group_number in range(start_group,len(batches)):\n                synchronize();start=time.monotonic()\n                count,loss,norm=optimizer_update(model,params,optimizer,scaler,batches[group_number],tokens,targets,pack)\n                synchronize();elapsed=time.monotonic()-start\n                stage_rows+=count;local_step+=1;progress[\'steps\']+=1\n                progress.update(phase=phase,epoch=epoch+1,epochs=epochs,update_in_phase=local_step,\n                    stage_row_exposures=stage_rows,finite_gradients=True,scaler_scale=scaler.get_scale(),\n                    last_loss=loss,last_grad_norm=norm,last_update_rows=count,last_update_seconds=elapsed)\n                pause=time.monotonic()-began>args.max_seconds-180\n                if local_step%25==0 or pause:\n                    saved=save_weights(\'RESUME_ONLY\',dict(stage=\'p3_resume_only\',phase=phase,\n                        official_fit_rows=0 if phase==\'external_old4404\' else len(indices),\n                        own_synthetic_fit_rows=4404,official_adaptation_completed=False))\n                    state=dict(phase=phase,epoch_zero_based=epoch,next_group=group_number+1,\n                        optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),seed=seed,epochs=epochs,\n                        lora_lr=lora_lr,head_lr=head_lr,indices=list(map(int,indices)),trainable_names=list(params),\n                        stage_rows=stage_rows,local_step=local_step,total_steps=progress[\'steps\'],\n                        protocol_sha256=protocol_sha,adapter_sha256=sha(saved),cpu_rng=torch.get_rng_state(),\n                        cuda_rng=torch.cuda.get_rng_state_all())\n                    temp=out/\'RESUME_ONLY.pt.tmp\';torch.save(state,temp);temp.replace(out/\'RESUME_ONLY.pt\')\n                    write();print(\'P3_TRAIN_PROGRESS\',json.dumps(progress),flush=True)\n                if pause:raise BudgetPause(\'Exact private adapter/AdamW/scaler/cursor saved; no repeated updates\')\n            progress[\'last_completed_epoch\']=dict(phase=phase,epoch=epoch+1,rows=len(indices));write()\n        assert stage_rows==len(indices)*epochs,(phase,stage_rows,len(indices),epochs)\n        for param in params.values():param.grad=None\n\n    def predict(tokens,indices):\n        model.model.eval();model.norms.eval();model.plain.eval();result={}\n        lengths=np.asarray([len(t)+1 for t in tokens])\n        with torch.no_grad():\n            for rows in bucket_batches(indices,lengths,0,False,microbatch=8):\n                if time.monotonic()-began>args.max_seconds-60:raise BudgetPause(\'Phase adapter saved; resume validation without refitting\')\n                scores=forward_logits(model,pack(tokens,rows)).cpu().numpy()\n                result.update({i:s for i,s in zip(rows,scores)})\n        return np.asarray([result[int(i)] for i in indices])\n\n    try:\n        assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))\n        assert sha(args.train)==protocol[\'train_sha256\'] and sha(args.synthetic)==protocol[\'old4404_csv_sha256\']\n        assert protocol[\'source_revision\']==REVISION and protocol[\'validation_metrics_observed\'] is False\n        official=pd.read_csv(args.train,dtype=str).fillna(\'\');synthetic=pd.read_csv(args.synthetic,dtype=str).fillna(\'\')\n        assert len(official)==2699 and len(synthetic)==4404 and synthetic.groupby(\'prompt_id\').size().eq(12).all()\n        for fold in args.folds:\n            spec=protocol[\'folds\'][str(fold)];fit=spec[\'fit\'];valid=spec[\'valid\']\n            assert len(set(fit)|set(valid))==2159 and not set(fit)&set(valid)\n            assert not set(official.iloc[fit].prompt_id)&set(official.iloc[valid].prompt_id)\n        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction=False\n        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction=False\n        model=Qwen32Classifier(args.base,sorted(official.model.unique()),seed=protocol[\'seed\'])\n        params=enable_training(model);frozen=parameter_sample(model)\n        assert sum(p.numel() for p in params.values())==17596434\n        progress.update(trainable_parameters=17596434,backbone_frozen=True,benchmark_weights_used=False)\n        if previous and (previous/\'progress.json\').exists():\n            prior=json.loads((previous/\'progress.json\').read_text());assert prior[\'protocol_sha256\']==protocol_sha\n            progress[\'steps\']=prior[\'steps\']\n            if (previous/\'RESUME_ONLY.pt\').exists():resume=torch.load(previous/\'RESUME_ONLY.pt\',map_location=\'cpu\',weights_only=False)\n        tokens=model.tokenizer(official.text.tolist(),add_special_tokens=False,truncation=False)[\'input_ids\']\n        extra=model.tokenizer(synthetic.text.tolist(),add_special_tokens=False,truncation=False)[\'input_ids\']\n        assert max(map(len,tokens+extra))+1<=model.model.config.max_position_embeddings\n        targets=np.asarray([model.labels.index(x) for x in official.model]);extra_targets=np.asarray([model.labels.index(x) for x in synthetic.model])\n        recipe=protocol[\'recipe\'];external=previous/\'external_old4404.safetensors\' if previous else None\n        if external and external.exists():\n            meta=check_adapter(external,\'p3_external_old4404\');assert meta[\'official_fit_rows\']==0 and meta[\'own_synthetic_fit_rows\']==4404\n            external=preserve(external)\n            progress[\'external_reused\']=True\n        else:\n            train(\'external_old4404\',extra,extra_targets,list(range(4404)),1,\n                recipe[\'external_lora_lr\'],recipe[\'external_norm_head_lr\'],protocol[\'seed\'])\n            external=save_weights(\'external_old4404\',dict(stage=\'p3_external_old4404\',official_fit_rows=0,\n                official_adaptation_completed=False,own_synthetic_fit_rows=4404,own_external_epochs=1))\n        progress.update(external_adapter_sha256=sha(external));progress[\'completed\'].append(\'external_old4404\');write()\n        results=[]\n        for fold in args.folds:\n            spec=protocol[\'folds\'][str(fold)];fit=spec[\'fit\'];valid=spec[\'valid\'];phase=f\'fold{fold}_fresh32\'\n            trained=previous/(phase+\'.safetensors\') if previous else None\n            if trained and trained.exists():\n                meta=check_adapter(trained,\'p3_cv_official\');assert meta[\'fit_indices\']==fit and meta[\'valid_indices\']==valid\n                trained=preserve(trained);model.reload(trained);progress[\'completed_adapter_reused_for_validation\']=phase\n            else:\n                model.reload(external)\n                train(phase,tokens,targets,fit,2,recipe[\'official_lora_lr\'],recipe[\'official_norm_head_lr\'],protocol[\'seed\']+1000*fold)\n                trained=save_weights(phase,dict(stage=\'p3_cv_official\',fold=fold,official_fit_rows=len(fit),\n                    own_synthetic_fit_rows=4404,official_adaptation_completed=True,official_epochs=2,fit_indices=fit,valid_indices=valid))\n            scores=predict(tokens,valid);before=predict(tokens,valid[:3])\n            with torch.no_grad():model.plain.bias.add_(.1)\n            model.reload(trained);after=predict(tokens,valid[:3]);reload_error=float(np.abs(before-after).max());assert reload_error<.002\n            np.savez_compressed(out/(phase+\'_oof.npz\'),indices=np.asarray(valid),ids=official.iloc[valid].id.to_numpy(dtype=str),labels=np.asarray(model.labels),logits=scores)\n            result=dict(fold=fold,standalone32_accuracy=float((scores.argmax(1)==targets[valid]).mean()),rows=len(valid),\n                reload_error=reload_error,adapter_sha256=sha(trained),primary_mixture_quality_not_yet_decided=True)\n            results.append(result);progress[\'completed\'].append(phase);progress[\'fold_results\']=results;write()\n            print(\'P3_STANDALONE_FOLD_RESULT\',json.dumps(result),flush=True)\n        assert all(torch.equal(v,parameter_sample(model)[n]) for n,v in frozen.items())\n        progress.update(status=\'P3_FRESH32_PART_COMPLETE\',frozen_parameters_unchanged=True,quality_measured=True,\n            full_ensemble_cv=False,primary_mixture_quality_not_yet_decided=True,no_official_attempt=True);write()\n    except BudgetPause as error:\n        progress.update(status=\'P3_BUDGET_PAUSED_CHECKPOINT_SAVED\',reason=str(error));write()\n    except BaseException as error:\n        progress.update(status=\'P3_ACTUAL_FAILED\',error=str(error),error_type=type(error).__name__,traceback=traceback.format_exc());write();raise\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in [\'base\',\'synthetic\',\'train\',\'protocol\',\'output\']:p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--previous-output\');p.add_argument(\'--max-seconds\',type=float,default=14400)\n    p.add_argument(\'--folds\',nargs=\'+\',type=int,default=[0]);run(p.parse_args())\n')
    (root/'p3_training_utils.py').write_text("import hashlib\nfrom pathlib import Path\nimport torch\nfrom torch.utils.checkpoint import checkpoint\nfrom fulltext_pooling import MeanPoolHooks\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef wanted_weights(model):\n    result = {f'backbone.{n}': p for n, p in model.model.named_parameters()\n              if '.lora_A' in n or '.lora_B' in n}\n    for prefix, module in [('norms', model.norms), ('plain', model.plain),\n                           ('centered', model.centered)]:\n        result.update({f'{prefix}.{n}': p for n, p in module.named_parameters()})\n    return result\n\ndef enable_training(model, checkpoint_tail=True):\n    # The original loader is inference-only. Freeze every non-LoRA backbone\n    # parameter, then explicitly enable the small set the optimizer may update.\n    for p in model.model.parameters():\n        p.requires_grad_(False)\n    trainable = {}\n    for name, p in model.model.named_parameters():\n        if '.lora_A' in name or '.lora_B' in name:\n            layer = int(name.split('.')[1])\n            assert 56 <= layer < 64 and p.dtype == torch.float32, name\n            p.requires_grad_(True)\n            trainable['backbone.' + name] = p\n    for prefix, module in [('norms', model.norms), ('plain', model.plain)]:\n        for name, p in module.named_parameters():\n            p.requires_grad_(True)\n            trainable[prefix + '.' + name] = p\n    # The scored inference route uses only the plain head. Keep the unused\n    # centered head unchanged rather than fit it with invalid singleton groups.\n    model.centered.requires_grad_(False)\n    assert trainable and all(p.requires_grad for p in trainable.values())\n    backbone_enabled = {n for n, p in model.model.named_parameters() if p.requires_grad}\n    assert backbone_enabled == {n.removeprefix('backbone.') for n in trainable\n                                if n.startswith('backbone.')}\n    if checkpoint_tail:\n        # Wrap the layer's forward, not __call__. Mean-pooling hooks therefore\n        # fire once, outside checkpoint recomputation, and keep no token states.\n        for layer in model.model.layers[56:64]:\n            original = layer.forward\n\n            def forward(*args, _original=original, _layer=layer, **kwargs):\n                if torch.is_grad_enabled() and _layer.training:\n                    return checkpoint(_original, *args, use_reentrant=False,\n                                      preserve_rng_state=True, **kwargs)\n                return _original(*args, **kwargs)\n\n            layer.forward = forward\n    return trainable\n\ndef forward_logits(model, pack):\n    with MeanPoolHooks(model.model, model.depths, model.norms,\n                       pack['attention_mask']) as pooled:\n        with torch.autocast('cuda', dtype=torch.float16):\n            model.model(inputs_embeds=model.model.embed_tokens(pack['input_ids']),\n                        attention_mask=pack['attention_mask'],\n                        output_hidden_states=False, use_cache=False)\n        features = pooled.features()\n        logits = model.plain(features)\n    assert features.shape == (pack['input_ids'].shape[0], 40960)\n    assert torch.isfinite(features).all() and torch.isfinite(logits).all()\n    return logits.float()\n\ndef synchronize():\n    for i in range(torch.cuda.device_count()):\n        torch.cuda.synchronize(i)\n\ndef parameter_sample(model):\n    # Fixed frozen parameters are enough to detect accidental optimizer inclusion;\n    # exact optimizer membership is separately asserted above.\n    result = {}\n    for n, p in model.model.named_parameters():\n        if not p.requires_grad and p.numel():\n            result[n] = p.detach().flatten()[:16].cpu().clone()\n    return result\n")
    (root/'pipeline.py').write_text('"""Offline genuine train32 + nine-V5 + spectral pipeline;8.5h hard target."""\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\ndef run(config):\n    start=float(os.environ[\'MERGE26_STARTED_WALL\'])\n    deadline=start+30600\n    root=Path(__file__).resolve().parent\n    work=Path(\'/kaggle/working\');data=Path(config[\'data\']);asset=Path(config[\'asset\'])\n    env=dict(os.environ,PYTHONPATH=str(root),NTL_DROP_Q35=\'1\',\n        HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',HF_HUB_DISABLE_TELEMETRY=\'1\',\n        USE_HUB_KERNELS=\'NO\',TOKENIZERS_PARALLELISM=\'false\',\n        MERGE26_SVC_BANK=str(asset/\'svc_cache\'),\n        MERGE26_SPECTRAL_FEATURES=\'/tmp/merge26-spectral.npy\')\n    def remaining():return deadline-time.time()\n    def child(command,environment=None,reserve=180):\n        assert remaining()>reserve+60,\'Insufficient global end-to-end budget\'\n        subprocess.run([sys.executable,\'-u\',*map(str,command)],cwd=root,\n            env=environment or env,check=True,timeout=remaining()-reserve)\n    # Only32B gets its exact validated offline package target. Parent nlm\n    # dependencies remain in their original environment and separate process.\n    target=root/\'qwen32-python\'\n    wheels=[asset/\'wheels\'/n for n in config[\'wheels\']]\n    assert all(p.is_file() for p in wheels)\n    subprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'--no-index\',\'--no-deps\',\n        \'--disable-pip-version-check\',\'--target\',str(target),*map(str,wheels)],check=True,timeout=180)\n    from input_stage import staged_dataset_tree\n    model=Path(\'/kaggle/input/datasets/kirilltym/ntl-qwen3-32b-nf4-pinned/p3_qwen32_nf4_base\')\n    trained=asset/\'fresh32_final.safetensors\'\n    with staged_dataset_tree(model,config[\'model_files\'],label=\'merge26-qwen32\') as staged:\n        qenv=dict(env,PYTHONPATH=str(target)+os.pathsep+str(root))\n        args=[root/\'train32_worker.py\',\'--base\',staged,\'--external\',asset/\'data/external_all_unfiltered.csv\',\n            \'--train\',data/\'train.csv\',\'--query\',data/\'test.csv\',\'--recipe\',asset/\'data/recipe32.json\',\n            \'--output\',work/\'merge26_32\',\'--deadline-wall\',str(deadline),\n            \'--train-max-seconds\',\'12600\']\n        if trained.exists():args+=[\'--trained-adapter\',trained]\n        child(args,qenv,reserve=config[\'mandatory_nlm_reserve_seconds\'])\n    child([root/\'setup_nlm.py\'],reserve=1800)\n    nlm=json.loads((root/\'nlm_paths.json\').read_text())\n    assert all(len(s[\'adapters\'])==3 for s in nlm[\'models\'][:3])\n    from restore_public24 import restore\n    recovery=Path(\'/kaggle/input/datasets/fleongg/ntl-legacy-qwen24-recovery\')\n    env[\'MERGE26_PUBLIC24\']=str(restore(recovery))\n    env[\'MERGE26_PUBLIC24_EXPORT_ONLY\']=\'1\'\n    cstart=time.time()\n    child([root/\'comp_stack.py\'],reserve=7200)\n    print(\'MERGE26_COMP_SECONDS\',time.time()-cstart,flush=True)\n    # One exact old public24 forward pass exports features for the old bank;\n    # it does not add a formerly skipped member to nlm\'s complementary mix.\n    helper=Path(\'/kaggle/input/datasets/kirilltym/ntl-spectral-style-heads-paired-4404/night24_readout_banks\')\n    child([root/\'night24_cpu_worker.py\',\'--features\',\'/tmp/merge26-spectral.npy\',\n        \'--feature-metadata\',\'/tmp/merge26-spectral.json\',\'--query\',data/\'test.csv\',\n        \'--bank\',helper/\'synthetic4404.joblib\',\'--bank-metadata\',helper/\'COMPLETE.json\',\n        \'--out\',work/\'merge26-spectral.npz\'],dict(env,CUDA_VISIBLE_DEVICES=\'\'),reserve=1800)\n    env[\'MERGE26_FUSION_PATHS\']=json.dumps(dict(spectral4404=str(work/\'merge26-spectral.npz\'),\n        fresh32=str(work/\'merge26_32/fresh32_logits.npz\')))\n    # infer.py T0 is the original notebook start, so its ATLAS projection\n    # includes training, I/O and the complement, not just its own process.\n    child([root/\'infer.py\',\'--data\',data,\'--models\',json.dumps(nlm[\'models\']),\n        \'--perpassage\',\'1\',\'--extra\',nlm[\'extra\'],\'--extra_w\',\'.5\',\'--tfidf_w\',\'0\',\n        \'--distort_w\',\'0\',\'--svc_w\',\'1.5\',\'--pool\',nlm[\'pool\'],\'--adapt_w\',\'1\',\n        \'--adapt_thr\',\'.15\',\'--sinkhorn\',\'1\',\'--post\',\'1\',\n        \'--atlas_dir\',nlm[\'atlas\'],\'--atlas_base\',nlm[\'atlas_base\'],\n        \'--atlas_w\',\'.10\',\'--atlas_margin\',\'.3\',\'--atlas_bs\',\'4\',\'--atlas_chunk\',\'3\',\n        \'--atlas_budget\',str(30600-600),\'--comp_path\',work/\'comp_logit.npy\',\'--comp_w\',\'.15\',\n        \'--out\',work/\'submission.parquet\',\'--cache\',work/\'merge26_nlm_cache\'],reserve=120)\n    import pandas as pd\n    result=pd.read_parquet(work/\'submission.parquet\');query=pd.read_csv(data/\'test.csv\',dtype={\'id\':str})\n    assert list(result.columns)==[\'id\',\'model\'] and result.id.tolist()==query.id.tolist()\n    assert result.id.is_unique and result.model.isin(config[\'labels\']).all()\n    assert remaining()>0\n    report=dict(status=\'REAL_TRAIN32_AND_ENSEMBLE_COMPLETE\',rows=len(result),\n        elapsed_seconds=time.time()-start,model27_used=False,weights=dict(nlm=.7225,spectral4404=.1275,fresh32=.15),\n        V5_checkpoints=9,RAW_exported=False,preview=False)\n    (work/\'merge26_completion.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps(report),flush=True)\n')
    (root/'prefix32.py').write_text('"""Lossless frozen-prefix transport for Qwen3, not27B hybrid cache code.\n\nCache the actual hidden dtype, seven raw float32 pooled means and the exact\nprepared attention/RoPE kwargs. Apply TRAINABLE norms after loading. Cached\nmicrobatches are never repacked or sliced. Files stay private in /tmp.\n"""\nfrom contextlib import nullcontext\nimport torch\nfrom torch.nn import functional as F\n\n\nclass Captured(Exception):\n    pass\n\n\ndef move(value,device):\n    if isinstance(value,torch.Tensor):return value.detach().to(device=device)\n    if isinstance(value,dict):return {k:move(v,device) for k,v in value.items()}\n    if isinstance(value,tuple):return tuple(move(v,device) for v in value)\n    if isinstance(value,list):return [move(v,device) for v in value]\n    assert value is None or isinstance(value,(str,int,float,bool)),type(value)\n    return value\n\n\ndef capture(model,pack):\n    assert len(model.model.layers)==64 and model.depths==list(range(8,65,8))\n    assert all(not p.requires_grad for layer in model.model.layers[:56] for p in layer.parameters())\n    means={};saved={};handles=[]\n    mask=pack[\'attention_mask\'].unsqueeze(-1).float();den=mask.sum(1)\n    assert (den>0).all()\n    def mean_hook(depth):\n        def hook(module,inputs,output):\n            assert isinstance(output,torch.Tensor)\n            means[depth]=((output.float()*mask.to(output.device)).sum(1)/den.to(output.device)).detach().cpu()\n        return hook\n    def stop_hook(module,args,kwargs):\n        assert args and isinstance(args[0],torch.Tensor) and len(args)==1\n        saved.update(hidden=move(args[0],\'cpu\'),kwargs=move(kwargs,\'cpu\'))\n        raise Captured()\n    try:\n        for depth in model.depths[:-1]:\n            handles.append(model.model.layers[depth-1].register_forward_hook(mean_hook(depth)))\n        handles.append(model.model.layers[56].register_forward_pre_hook(stop_hook,with_kwargs=True))\n        with torch.no_grad():\n            context=torch.autocast(\'cuda\',dtype=torch.float16) if model.device.type==\'cuda\' else nullcontext()\n            with context:\n                model.model(inputs_embeds=model.model.embed_tokens(pack[\'input_ids\']),\n                    attention_mask=pack[\'attention_mask\'],output_hidden_states=False,use_cache=False)\n        raise AssertionError(\'Frozen prefix stop hook was not called\')\n    except Captured:\n        assert set(means)==set(model.depths[:-1])\n        saved[\'means\']=torch.stack([means[d] for d in model.depths[:-1]],dim=1)\n        saved[\'attention_mask\']=move(pack[\'attention_mask\'],\'cpu\')\n        saved[\'hidden_dtype\']=str(saved[\'hidden\'].dtype)\n        saved[\'prefix_depth\']=56\n        assert saved[\'means\'].dtype==torch.float32\n        return saved\n    finally:\n        for handle in handles:handle.remove()\n\n\ndef logits(model,cached):\n    assert cached[\'prefix_depth\']==56\n    hidden=move(cached[\'hidden\'],model.device)\n    assert str(hidden.dtype)==cached[\'hidden_dtype\']\n    kwargs=move(cached[\'kwargs\'],model.device)\n    means=move(cached[\'means\'],model.device)\n    mask=move(cached[\'attention_mask\'],model.device).unsqueeze(-1).float()\n    context=torch.autocast(\'cuda\',dtype=torch.float16) if model.device.type==\'cuda\' else nullcontext()\n    with context:\n        for layer in model.model.layers[56:]:\n            hidden=layer(hidden,**kwargs)\n            assert isinstance(hidden,torch.Tensor)\n    # Match MeanPoolHooks: norms/head outside the autocast context, pre-LN\n    # mean contiguous just as the original reduction result.\n    final_mean=((hidden.float()*mask.to(hidden.device)).sum(1)/mask.sum(1).to(hidden.device)).to(model.device)\n    context=torch.autocast(\'cuda\',enabled=False) if model.device.type==\'cuda\' else nullcontext()\n    with context:\n        values=[norm(means[:,i].contiguous()) for i,norm in enumerate(model.norms[:-1])]\n        values.append(model.norms[-1](final_mean))\n        scores=model.plain(torch.cat(values,dim=-1))\n    assert torch.isfinite(scores).all()\n    return scores.float()\n\n\ndef gradient_parity(model,params,pack,targets,full_forward):\n    """Before any update, compare cache with two full passes\' FP16 floor.\n\n    Fixed guard BEFORE metrics: logits <=2e-4; grad relativeL2 <=1e-4\n    AND <=max(1e-5,2*full_repeat+1e-6). No optimizer update, no relaxed retry.\n    """\n    def one(cached=None):\n        for p in params.values():p.grad=None\n        scores=full_forward(model,pack) if cached is None else logits(model,cached)\n        loss=F.cross_entropy(scores,targets)*128.\n        loss.backward()\n        grads={n:p.grad.detach().cpu().clone() for n,p in params.items()}\n        assert all(torch.isfinite(g).all() for g in grads.values())\n        return scores.detach().cpu(),grads\n    def relative(a,b):\n        num=sum(float((a[n].double()-b[n].double()).square().sum()) for n in a)\n        den=sum(float(a[n].double().square().sum()) for n in a)\n        return (num/max(den,1e-30))**.5\n    first,g1=one();repeat,g2=one()\n    cached=capture(model,pack);actual,gc=one(cached)\n    logit_error=float((first-actual).abs().max())\n    floor=relative(g1,g2);error=relative(g1,gc)\n    limit=max(1e-5,2*floor+1e-6)\n    report=dict(logit_max_error=logit_error,all_trainable_gradient_relative_L2=error,\n        full_repeat_gradient_relative_L2=floor,relative_limit=min(1e-4,limit),\n        cached_hidden_dtype=cached[\'hidden_dtype\'],raw_mean_dtype=str(cached[\'means\'].dtype),\n        cached_layout_preserved=True,optimizer_updates=0)\n    for p in params.values():p.grad=None\n    assert logit_error<=2e-4 and error<=1e-4 and error<=limit,report\n    return report\n')
    (root/'qwen32_nf4.py').write_text('"""Fresh Qwen3-32B NF4 classifier; NEVER loads the Qwen3.8-27B adapter.\n\nPacked-linear/host-embedding utilities are credited to lightningv08\'s NF4\nloader. This uses the official Qwen3 decoder, its own pinned base/tokenizer,\nnewly initialized rank16 tail-eight LoRA and independent readouts.\n"""\nimport hashlib\nimport json\nimport math\nimport struct\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom accelerate.utils import set_module_tensor_to_device\nfrom bitsandbytes.functional import QuantState\nfrom safetensors.torch import load_file\nfrom transformers import AutoTokenizer\nfrom transformers.models.qwen3.configuration_qwen3 import Qwen3Config\nfrom transformers.models.qwen3 import modeling_qwen3 as qmodel\nfrom nf4_linear_common import NF4InferenceLinear, HostEmbedding\n\nREPOSITORY=\'unsloth/Qwen3-32B-bnb-4bit\'\nREVISION=\'7f721e74a6a8cc9ee352f7e49303a2c1705f9083\'\nEOS=151645\nDEPTHS=list(range(8,65,8))\n\n\ndef validate_metadata(metadata):\n    assert metadata[\'backbone_family\']==\'Qwen3-32B\', \'Wrong decoder family\'\n    assert metadata[\'source_repository\']==REPOSITORY and metadata[\'source_revision\']==REVISION\n    assert metadata[\'native_decoder_layers\']==metadata[\'executed_decoder_layers\']==64\n    assert metadata[\'lora_target_layer_indices\']==list(range(56,64))\n    assert metadata[\'rank\']==16 and metadata[\'alpha\']==32\n    assert metadata[\'depths\']==DEPTHS and metadata[\'hidden_size\']==5120\n    assert metadata[\'classification_token_id\']==EOS\n    assert len(metadata[\'labels\'])==len(set(metadata[\'labels\']))==18\n    assert metadata[\'text_truncation\'] is False and metadata[\'inference_max_length\'] is None\n\n\nclass Qwen32Classifier:\n    def __init__(self,base,labels,adapter=None,seed=20261025):\n        start=time.perf_counter()\n        assert torch.cuda.device_count()==2\n        self.base=Path(base);self.device=torch.device(\'cuda:0\')\n        self.dtype=torch.float16\n        self.depths=DEPTHS.copy();self.labels=list(labels)\n        assert len(self.labels)==len(set(self.labels))==18\n        provenance=json.loads((self.base/\'PINNED_SOURCE.json\').read_text())\n        assert provenance[\'repository\']==REPOSITORY and provenance[\'revision\']==REVISION\n        config=json.loads((self.base/\'config.json\').read_text())\n        assert config[\'model_type\']==\'qwen3\' and config[\'eos_token_id\']==EOS\n        assert config[\'num_hidden_layers\']==64 and config[\'hidden_size\']==5120\n        assert config[\'num_attention_heads\']==64 and config[\'num_key_value_heads\']==8\n        assert config[\'head_dim\']==128 and config[\'attention_dropout\']==0.\n        assert config[\'quantization_config\'][\'bnb_4bit_quant_type\']==\'nf4\'\n        assert config[\'quantization_config\'][\'bnb_4bit_use_double_quant\'] is True\n        cfg=Qwen3Config(**config);cfg._attn_implementation=\'sdpa\';cfg.use_cache=False\n        torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n        torch.set_num_threads(8)\n        with torch.device(\'meta\'):self.model=qmodel.Qwen3Model(cfg)\n        # Pool pre-final-normalization states at all eight declared depths.\n        self.model.norm=nn.Identity()\n        self.index=json.loads((self.base/\'model.safetensors.index.json\').read_text())[\'weight_map\']\n        self.headers,self.maps={},{}\n        count=0\n        for name,module in list(self.model.named_modules()):\n            if not isinstance(module,nn.Linear):continue\n            layer=int(name.split(\'.\')[1]);device=torch.device(f\'cuda:{layer*2//64}\')\n            key=\'model.\'+name+\'.weight\'\n            state=QuantState.from_dict({k:self.tensor(k) for k in self.index\n                                       if k.startswith(key+\'.\')},device=device)\n            assert tuple(state.shape)==(module.out_features,module.in_features),(key,state.shape)\n            assert module.bias is None\n            rank=16 if layer>=56 else 0\n            replacement=NF4InferenceLinear(self.tensor(key),state,\n                (module.in_features,module.out_features),rank,device,True)\n            if rank:\n                with torch.no_grad():\n                    nn.init.kaiming_uniform_(replacement.lora_A,a=math.sqrt(5))\n                    replacement.lora_B.zero_()\n            parent,_,attribute=name.rpartition(\'.\')\n            setattr(self.model.get_submodule(parent),attribute,replacement)\n            count+=1\n        assert count==448\n        # T4 portable execution: FP16 embeddings and unquantized norm weights,\n        # not the upstream BF16 compute setting. No Qwen3.5 hybrid modules.\n        self.model.embed_tokens=HostEmbedding(self.tensor(\'model.embed_tokens.weight\').to(torch.float16))\n        self.model.rotary_emb=qmodel.Qwen3RotaryEmbedding(cfg,device=self.device)\n        for name,param in self.model.named_parameters():\n            if \'.lora_A\' in name or \'.lora_B\' in name:continue\n            value=self.tensor(\'model.\'+name)\n            set_module_tensor_to_device(self.model,name,self.device,value=value,\n                                        dtype=torch.float16 if value.is_floating_point() else value.dtype)\n        assert not any(p.is_meta for p in self.model.parameters())\n        assert not any(b.is_meta for b in self.model.buffers())\n        self.norms=nn.ModuleList([nn.LayerNorm(5120) for _ in self.depths]).to(self.device)\n        self.plain=nn.Linear(40960,18).to(self.device)\n        self.centered=nn.Linear(40960,18).to(self.device)\n        self.tokenizer=AutoTokenizer.from_pretrained(self.base,local_files_only=True)\n        self.tokenizer.pad_token_id=EOS;self.tokenizer.padding_side=\'right\'\n        self.metadata=dict(backbone_family=\'Qwen3-32B\',source_repository=REPOSITORY,\n            source_revision=REVISION,native_decoder_layers=64,executed_decoder_layers=64,\n            hidden_size=5120,depths=self.depths,rank=16,alpha=32,\n            lora_target_layer_indices=list(range(56,64)),labels=self.labels,\n            classification_token_id=EOS,text_truncation=False,inference_max_length=None,\n            portable_compute_dtype=\'float16\',head_dtype=\'float32\',fresh_adapter_seed=seed,\n            fresh_adapter=True,official_fit_rows=0,own_synthetic_fit_rows=0,\n            benchmark_only=True,not_for_scoring=True)\n        validate_metadata(self.metadata)\n        if adapter:self.reload(adapter)\n        self.model.eval();self.norms.eval();self.plain.eval();self.centered.eval()\n        print(\'P3_QWEN32_LOADED\',json.dumps(dict(seconds=time.perf_counter()-start,\n            linears=count,layers=64,revision=REVISION,EOS=EOS,portable=\'FP16\',\n            fresh_adapter=adapter is None,uses27B_adapter=False)),flush=True)\n\n    def tensor(self,key):\n        path=self.base/self.index[key]\n        if path not in self.headers:\n            with path.open(\'rb\') as stream:\n                size=struct.unpack(\'<Q\',stream.read(8))[0];assert size<32*2**20\n                self.headers[path]=(8+size,json.loads(stream.read(size)))\n            self.maps[path]=np.memmap(path,dtype=np.uint8,mode=\'r\')\n        offset,header=self.headers[path];entry=header[key];lo,hi=entry[\'data_offsets\']\n        dtype={\'U8\':torch.uint8,\'F16\':torch.float16,\'BF16\':torch.bfloat16,\n               \'F32\':torch.float32,\'I64\':torch.int64}[entry[\'dtype\']]\n        assert hi-lo==math.prod(entry[\'shape\'])*torch.empty((),dtype=dtype).element_size()\n        return torch.frombuffer(self.maps[path][offset+lo:offset+hi],dtype=dtype).reshape(entry[\'shape\'])\n\n    def reload(self,path):\n        metadata=json.loads(Path(path).with_suffix(\'.json\').read_text())\n        validate_metadata(metadata)\n        assert metadata[\'labels\']==self.labels\n        saved=load_file(str(path),device=\'cpu\')\n        wanted={f\'backbone.{n}\':p for n,p in self.model.named_parameters()\n                if \'.lora_A\' in n or \'.lora_B\' in n}\n        for prefix,module in [(\'norms\',self.norms),(\'plain\',self.plain),(\'centered\',self.centered)]:\n            wanted.update({prefix+\'.\'+n:p for n,p in module.named_parameters()})\n        assert set(saved)==set(wanted),\'Incompatible fresh32B state; never substitute27B weights\'\n        with torch.no_grad():\n            for name,param in wanted.items():\n                assert saved[name].shape==param.shape and torch.isfinite(saved[name]).all(),name\n                param.copy_(saved[name])\n')
    (root/'readouts.py').write_text('"""Original prompt-centred readout ideas, refit on the exact public LoRA24 features.\n\nNo text or ground-truth test labels are stored in exported model banks.\n"""\nimport numpy as np\nfrom scipy.spatial.distance import pdist, squareform\nfrom scipy.cluster.hierarchy import linkage, fcluster\nfrom sklearn.preprocessing import normalize, StandardScaler\nfrom sklearn.random_projection import SparseRandomProjection\nfrom sklearn.decomposition import PCA\nfrom sklearn.kernel_approximation import RBFSampler, Nystroem\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.svm import SVC\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nHEAD_WEIGHTS = dict(rbf=.05, fourier=.075, laplace=.075, knn=.05,\n                    centered=.10, word_pca=.15, char_pca=.20, late=.30)\nLAYERS = [2, 4, 6, 8, 12, 16, 20, 24]\n\ndef probability(p):\n    p = np.asarray(p, np.float64)\n    assert p.ndim == 2 and p.shape[1] == 18 and np.isfinite(p).all()\n    p = np.maximum(p, 1e-12)\n    return p / p.sum(1, keepdims=True)\n\ndef residual(x, groups):\n    a = np.array(x, copy=True)\n    for g in np.unique(groups):\n        mask = groups == g\n        if 2 <= int(mask.sum()) <= 18:\n            a[mask] -= a[mask].mean(0)\n    return normalize(a).astype(np.float32)\n\ndef gamma(x, metric=\'sqeuclidean\'):\n    # Exact same official-fit median rule; no test-dependent bandwidth selection.\n    return 1 / max(float(np.median(pdist(x, metric))), 1e-6)\n\ndef weights(y, official_n):\n    w = np.ones(len(y), np.float64)\n    if len(y) > official_n:\n        w[official_n:] = .25 * official_n / (len(y) - official_n)\n    counts = np.bincount(y, weights=w, minlength=18)\n    return w * w.sum() / (18 * counts[y])\n\ndef logistic(c, x, y, w):\n    return LogisticRegression(C=c, max_iter=600, random_state=42).fit(x, y, sample_weight=w)\n\ndef pca_fit(x):\n    return PCA(n_components=min(64, x.shape[1], len(x)-1), svd_solver=\'randomized\', iterated_power=3, random_state=42).fit(x)\n\ndef fit_bank(z, y, groups, official_n, labels, adapter_sha):\n    z = np.asarray(z, np.float32); y = np.asarray(y, int); groups = np.asarray(groups, str)\n    assert z.shape[1] == 32768 and len(z) == len(y) == len(groups)\n    assert set(y[:official_n]) == set(range(18))\n    w = weights(y, official_n)\n    rp = SparseRandomProjection(n_components=512, dense_output=True, random_state=42).fit(z[:official_n])\n    x = normalize(rp.transform(z)).astype(np.float32)\n    reducer = pca_fit(x[:official_n]); pc = reducer.transform(x)\n    h = {}; gr = gamma(x[:official_n])\n    for name in [\'fourier\', \'laplace\']:\n        m = (RBFSampler(gamma=gr, n_components=1024, random_state=42) if name == \'fourier\'\n             else Nystroem(kernel=\'laplacian\', gamma=gamma(x[:official_n], \'cityblock\'), n_components=256, random_state=42, n_jobs=1))\n        a = m.fit_transform(x); h[name] = (m, logistic(10, a, y, w))\n    # Keep the neighbourhood geometry anchored on official data in both conditions.\n    sc = StandardScaler().fit(pc[:official_n]); a = sc.transform(pc[:official_n])\n    lda = LinearDiscriminantAnalysis(solver=\'eigen\', shrinkage=\'auto\', n_components=17).fit(a, y[:official_n])\n    knn = KNeighborsClassifier(n_neighbors=11, weights=\'distance\', n_jobs=1).fit(lda.transform(a), y[:official_n])\n    h[\'knn\'] = (sc, lda, knn)\n    margin = np.zeros((len(y), 18))\n    for ti, vi in GroupKFold(3).split(x, y, groups):\n        assert not set(groups[ti]) & set(groups[vi])\n        svc = SVC(C=3, gamma=gr, cache_size=256, decision_function_shape=\'ovr\').fit(x[ti], y[ti], sample_weight=w[ti])\n        margin[vi] = svc.decision_function(x[vi])\n    calibration = logistic(.1, margin, y, w)\n    h[\'rbf\'] = (SVC(C=3, gamma=gr, cache_size=256, decision_function_shape=\'ovr\').fit(x, y, sample_weight=w), calibration)\n    xc = residual(x, groups)\n    m = RBFSampler(gamma=gamma(xc[:official_n]), n_components=512, random_state=43)\n    h[\'centered\'] = (m, logistic(10, m.fit_transform(xc), y, w))\n    cp = pca_fit(xc[:official_n]); ac = normalize(cp.transform(xc))\n    for name, c in [(\'word_pca\', 10), (\'char_pca\', 3)]:\n        m = RBFSampler(gamma=gamma(ac[:official_n]), n_components=512, random_state=43)\n        h[name] = (cp, m, logistic(c, m.fit_transform(ac), y, w))\n    late_rp = []\n    blocks = []\n    for j in range(4, 8):\n        m = SparseRandomProjection(n_components=64, dense_output=True, random_state=42+j).fit(z[:official_n, j*4096:(j+1)*4096])\n        late_rp.append(m); blocks.append(normalize(m.transform(z[:, j*4096:(j+1)*4096])))\n    a = normalize(np.concatenate(blocks, 1)); a = residual(a, groups)\n    cp = pca_fit(a[:official_n]); a = normalize(cp.transform(a))\n    m = RBFSampler(gamma=gamma(a[:official_n]), n_components=512, random_state=43)\n    h[\'late\'] = (cp, m, logistic(3, m.fit_transform(a), y, w))\n    return dict(rp=rp, raw_pca=reducer, late_rp=late_rp, heads=h, labels=list(labels),\n                head_weights=HEAD_WEIGHTS, layers=LAYERS, adapter_sha256=adapter_sha,\n                official_rows=official_n, synthetic_rows=len(y)-official_n,\n                synthetic_total_weight_relative_to_official=.25 if len(y)>official_n else 0,\n                fit_scope=\'all official rows; synthetic rows only in augmented bank; no hidden test fit\',\n                adaptation=\'readout training only; base LoRA adapter unchanged\')\n\ndef infer_groups(texts, kind):\n    if len(texts) < 2: return np.arange(len(texts))\n    opts = dict(sublinear_tf=True, dtype=np.float32)\n    if kind == \'char\': opts.update(analyzer=\'char\', ngram_range=(3,5), max_features=100000)\n    else: opts.update(stop_words=\'english\')\n    try: tf = TfidfVectorizer(**opts).fit_transform(texts)\n    except ValueError: return np.arange(len(texts))\n    d = np.clip(1-(tf@tf.T).toarray(), 0, 2); np.fill_diagonal(d, 0)\n    return fcluster(linkage(squareform(d, checks=False), method=\'average\'), .85, criterion=\'distance\')\n\ndef predict_bank(bank, z, texts):\n    assert len(z) == len(texts) and z.shape[1] == 32768\n    x = normalize(bank[\'rp\'].transform(z)).astype(np.float32)\n    word = infer_groups(texts, \'word\'); chars = infer_groups(texts, \'char\')\n    h = bank[\'heads\']; out = {}\n    for name in [\'fourier\',\'laplace\']:\n        m, clf = h[name]; out[name] = probability(clf.predict_proba(m.transform(x)))\n    sc, lda, knn = h[\'knn\']; out[\'knn\'] = probability(knn.predict_proba(lda.transform(sc.transform(bank[\'raw_pca\'].transform(x)))))\n    svc, cal = h[\'rbf\']; out[\'rbf\'] = probability(cal.predict_proba(svc.decision_function(x)))\n    m, clf = h[\'centered\']; out[\'centered\'] = probability(clf.predict_proba(m.transform(residual(x, word))))\n    for name, g in [(\'word_pca\',word), (\'char_pca\',chars)]:\n        cp, m, clf = h[name]; a = normalize(cp.transform(residual(x, g))); out[name] = probability(clf.predict_proba(m.transform(a)))\n    blocks = [normalize(m.transform(z[:,j*4096:(j+1)*4096])) for j,m in zip(range(4,8),bank[\'late_rp\'])]\n    cp,m,clf=h[\'late\']; a=normalize(cp.transform(residual(normalize(np.concatenate(blocks,1)),chars)))\n    out[\'late\']=probability(clf.predict_proba(m.transform(a)))\n    return probability(sum(bank[\'head_weights\'][name]*p for name,p in out.items())), out\n')
    (root/'restore_public24.py').write_text('"""Restore the exact public 24-layer checkpoint required by the old4404 bank.\n\nPublic source: fleongg/ntl-legacy-qwen24-recovery V1 datasetVersion21066826.\nThe nlm-v27 saved log skipped this absent checkpoint; exporting its features\ntherefore adds one 24-layer forward pass. It must not silently change nlm\'s\nexisting complementary mixture, which excluded this member in that run.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef restore(source, target=\'/tmp/merge26-public24/lora24/final\'):\n    source, target = Path(source), Path(target)\n    manifest = json.loads((source/\'legacy_recovery_manifest.json\').read_text())\n    assert manifest[\'independently_verified_published_migration_manifest\']\n    assert target.resolve().is_relative_to(Path(\'/tmp\'))\n    for relative, record in manifest[\'files\'].items():\n        rel = Path(relative)\n        assert not rel.is_absolute() and \'..\' not in rel.parts\n        assert Path(record[\'flat_file\']).name == record[\'flat_file\']\n        origin = source/record[\'flat_file\']\n        assert sha(origin) == record[\'sha256\'], relative\n        destination = target/rel\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(origin, destination)\n        assert sha(destination) == record[\'sha256\'], relative\n    assert sha(target/\'adapter/adapter_model.safetensors\') == \'a6aff409ae2df897717875373d038635fe5012fbf93557c90aea3457fda0e7f8\'\n    assert sha(target/\'heads.safetensors\') == \'2176ade29e6eb97bd3a6c204f944959f9757a80c6f1ff475898c23bce60ecd96\'\n    assert (target/\'COMPLETE.json\').is_file()\n    assert (target/\'readouts.npz\').is_file()\n    return target\n')
    (root/'setup_nlm.py').write_text('# Name That LLM - full solution: fold-averaged two-stage LoRA24 (Qwen3-8B-Base, Qwen3-14B-Base, Qwen2.5-7B),\n# Nishant two-stage checkpoints, 4-view n-gram SVM on train+pool, shared-prompt Gaussian, adaptive centred twins,\n# 10% public ATLAS (source-conditioned likelihood, arXiv 2610.19583) before Sinkhorn balancing.\nimport glob, os, subprocess, json, sys\n\n_merge_original_glob=glob.glob\ndef _merge_scoped_glob(pattern,*args,**kwargs):\n    found=_merge_original_glob(pattern,*args,**kwargs)\n    if str(pattern).startswith(\'/kaggle/input\'):\n        found=[p for p in found if not p.startswith(\'/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets\')\n            and not p.startswith(\'/kaggle/input/datasets/kirilltym/ntl-qwen3-32b-nf4-pinned\')\n            and not p.startswith(\'/kaggle/input/datasets/kirilltym/ntl-llama31-8b-base-pinned\')\n            and not p.startswith(\'/kaggle/input/notebooks/kirilltym/ntl-free-proxy-data-pilot\')\n            and not p.startswith(\'/kaggle/input/datasets/kirilltym/ntl-private-unfiltered-2026-10-26\')]\n    return found\nglob.glob=_merge_scoped_glob\n\nwhl = glob.glob(\'/kaggle/input/**/bitsandbytes-*.whl\', recursive=True)\nif whl: print(subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'--no-index\', \'--no-deps\', \'-q\'] + whl, capture_output=True, text=True).returncode)\ncfgs = sorted(glob.glob(\'/kaggle/input/**/config.json\', recursive=True))\ndef mdir(*parts):\n    c = [os.path.dirname(x) for x in cfgs if all(p in x.lower() for p in parts)]\n    assert c, parts; return c[0]\ndef adir(name):\n    h = glob.glob(f\'/kaggle/input/**/{name}/heads.pt\', recursive=True); assert h, name; return os.path.dirname(h[0])\nDATA = next(p for p in [\'/kaggle/input/competitions/name-that-llm\', \'/kaggle/input/name-that-llm\'] if os.path.exists(p + \'/train.csv\'))\nHEADS = os.path.dirname(next(iter(glob.glob(\'/kaggle/input/**/heads/q3_8bx.npz\', recursive=True))))\nEXTRA = next(iter(glob.glob(\'/kaggle/input/**/nlm_matched_clean.parquet\', recursive=True)))\nPOOL = next(iter(glob.glob(\'/kaggle/input/**/nlm_pool_v5.parquet\', recursive=True)))\nB8, B14, B25 = mdir(\'/qwen-3/\', \'/8b-base/\'), mdir(\'/qwen-3/\', \'/14b-base/\'), mdir(\'/qwen2.5/\', \'/7b/\')\nI8, I14 = mdir(\'/qwen-3/\', \'/8b/\'), mdir(\'/qwen-3/\', \'/14b/\')\nA8 = [adir(f\'ts_q3b8_f{k}\') for k in range(5)] + [adir(\'ts_q3b8\')]\nA14 = [adir(f\'ts_q3b14_f{k}\') for k in (0, 2, 4)] + [adir(\'ts_q3b14\')]\nA25 = [adir(\'ts_q25_7b\')]\nprint(B8, B14, B25, I8, I14, A8, A14, A25, HEADS, EXTRA, POOL, sep=\'\\n\')\ndef nmeta(name):\n    h = glob.glob(f\'/kaggle/input/**/{name}/meta.json\', recursive=True); assert h, name\n    d = os.path.dirname(h[0]); c = json.load(open(h[0]))[\'cfg\']; return d, c\nN8, N8C = nmeta(\'lora8b_v1\'); N14, N14C = nmeta(\'lora14b_v1\')\nprint(\'nishant\', N8, N8C[\'keep\'], N8C[\'max_len\'], N14, N14C[\'keep\'], N14C[\'max_len\'])\nATLAS = os.path.dirname(next(iter(glob.glob(\'/kaggle/input/**/atlas_bundle.json\', recursive=True))))\nprint(\'atlas\', ATLAS)\nBL = \'/kaggle/input/notebooks/kirilltym/ntl-free-proxy-data-pilot/llama31_8b_base\'\nassert os.path.isfile(os.path.join(BL,\'config.json\'))\nAL = [adir(f\'ts_llama8_f{k}\') for k in (0, 2, 4)] + [adir(\'ts_llama8\')]\nprint(\'llama\', BL, AL)\nA8S7 = [adir(f\'ts8_s7_f{k}\') for k in (0, 2, 4)] + [adir(\'ts8_s7\')]\nprint(\'seed7\', A8S7)\nB25_14 = mdir(\'/qwen2.5/\', \'/14b/\')\nN14B, N14BC = nmeta(\'lora_qwen3_14b\'); Q25_14, Q25_14C = nmeta(\'lora_qwen25_14b\')\nDEB = os.path.dirname(glob.glob(\'/kaggle/input/**/deberta_v3_large/meta.json\', recursive=True)[0])\nimport shutil; shutil.copytree(DEB, \'/kaggle/working/deberta_v3_large\', dirs_exist_ok=True); DEB = \'/kaggle/working/deberta_v3_large\'\nprint(\'part two\', N14B, N14BC[\'keep\'], N14BC[\'layers\'], N14BC.get(\'eos_id\'), Q25_14, Q25_14C[\'keep\'], Q25_14C[\'layers\'], DEB)\nAQ8V5 = [adir(f\'q8v5_f{k}\') for k in range(5)] + [adir(\'q8v5\')]\nprint(\'q8v5\', AQ8V5)\nAQ14V5 = [adir(f\'q14v5_f{k}\') for k in range(5)] + [adir(\'q14v5\')]\nALL8V5 = [adir(f\'ll8v5_f{k}\') for k in range(5)] + [adir(\'ll8v5\')]\nprint(\'v5\', AQ14V5, ALL8V5)\n\nW_N14, W_N8 = 1.0, 0.5\nL = [2, 4, 6, 8, 12, 16, 20, 24]\nMODELS = [\n  {"name": "q14v5", "type": "l24", "path": B14, "adapters": AQ14V5, "keep": 27, "layers": L, "w": 3.0, "bs": 8, "max_len": 384, "twin": 0.35, "gauss": 0.70},\n  {"name": "q8v5", "type": "l24", "path": B8, "adapters": AQ8V5, "keep": 24, "layers": L, "w": 2.0, "bs": 16, "max_len": 384, "twin": 0.35, "gauss": 0.44},\n  {"name": "ll8v5", "type": "l24", "path": BL, "adapters": ALL8V5, "keep": 24, "layers": L, "w": 1.0, "bs": 16, "max_len": 384, "twin": 0.35},\n  {"name": "n14b", "type": "l24", "path": B14, "adapters": [N14B], "keep": N14BC[\'keep\'], "layers": N14BC[\'layers\'], "w": 2.5, "bs": 8, "max_len": N14BC[\'max_len\'], "twin": 0.35, "no_final_norm": True, "eos_id": N14BC[\'eos_id\']},\n  {"name": "q25_14", "type": "l24", "path": B25_14, "adapters": [Q25_14], "keep": Q25_14C[\'keep\'], "layers": Q25_14C[\'layers\'], "w": 1.0, "bs": 8, "max_len": Q25_14C[\'max_len\'], "twin": 0.35, "no_final_norm": True, "eos_id": Q25_14C[\'eos_id\']},\n  {"name": "deb", "type": "enc", "ckpt": DEB, "w": 1.0},\n]\n\nfor s in MODELS[:3]:\n    name=s[\'name\']\n    s[\'adapters\']=[p for p in s[\'adapters\'] if os.path.basename(p) in [name,name+\'_f0\',name+\'_f2\']]\n    assert len(s[\'adapters\'])==3\njson.dump(dict(models=MODELS,extra=EXTRA,pool=POOL,atlas=ATLAS,atlas_base=B8),open(\'nlm_paths.json\',\'w\'))\n')
    (root/'svc_cache.py').write_text('"""Portable exact four-view SVC state; CPU fit and prediction only.\nRecipe credited to the public nlm-v27/fleongg stack. No pickle estimator.\n"""\nimport json\nimport re\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.sparse import hstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nVIEWS=[(\'char\',3,5,250000,1.),(\'word\',1,3,180000,.7),\n       (\'shape\',2,6,60000,.5),(\'function\',1,3,50000,.5)]\nFW=set(\'a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little\'.split())\n\ndef prepare(text,view):\n    if view==\'shape\':return re.sub(r\'[^\\W\\d_]+\',lambda m:\'X\' if m[0].isupper() else \'x\',text)\n    if view==\'function\':return \' \'.join(w.lower() if w.lower() in FW or not w.isalnum()\n        else \'CONTENT\' for w in re.findall(r\'\\w+|[^\\w\\s]\',text))\n    return text\n\ndef options(view,lo,hi,mf):\n    analyzer=\'char\' if view==\'shape\' else \'word\' if view==\'function\' else view\n    d=dict(analyzer=analyzer,ngram_range=(lo,hi),lowercase=False,sublinear_tf=True,\n        min_df=2,max_features=mf,dtype=np.float32)\n    if analyzer==\'word\':d[\'token_pattern\']=r\'(?u)\\b\\w+\\b|[^\\w\\s]\'\n    return d\n\ndef save_vectorizer(path,v,view,weight):\n    p=Path(path);params=v.get_params()\n    keys=[\'analyzer\',\'ngram_range\',\'lowercase\',\'sublinear_tf\',\'min_df\',\'max_features\',\'token_pattern\']\n    record=dict(params={k:params[k] for k in keys},vocabulary={k:int(i) for k,i in v.vocabulary_.items()},\n        view=view,weight=weight)\n    p.with_suffix(\'.json\').write_text(json.dumps(record,ensure_ascii=False))\n    np.save(p.with_suffix(\'.idf.npy\'),v.idf_,allow_pickle=False)\n\ndef matrix(folder,texts):\n    values=[]\n    for i in range(4):\n        p=Path(folder)/f\'view{i}\';r=json.loads(p.with_suffix(\'.json\').read_text())\n        opts=r[\'params\'];opts[\'ngram_range\']=tuple(opts[\'ngram_range\']);opts[\'dtype\']=np.float32\n        v=TfidfVectorizer(vocabulary=r[\'vocabulary\'],**opts)\n        v.idf_=np.load(p.with_suffix(\'.idf.npy\'),allow_pickle=False)\n        values.append(v.transform([prepare(t,r[\'view\']) for t in texts])*r[\'weight\'])\n    return hstack(values,format=\'csr\')\n\ndef predict_cache(folder,texts,labels):\n    meta=json.loads((Path(folder)/\'metadata.json\').read_text());assert meta[\'labels\']==list(labels)\n    z=np.load(Path(folder)/\'svc.npz\',allow_pickle=False)\n    assert np.array_equal(z[\'classes\'],np.arange(18))\n    values=np.asarray(matrix(folder,texts)@z[\'coef\'].T)+z[\'intercept\']\n    values-=values.max(axis=1,keepdims=True)\n    return values-np.log(np.exp(values).sum(axis=1,keepdims=True))\n')
    (root/'train32_worker.py').write_text('"""Genuine fresh32 train-and-predict worker; no dummy submission or27B state.\n\nOrdinary execution requires the user\'s per-run quota permission. The economy\nnotebook invokes this only after the exact public fixture preview has exited.\nNo GPU execution is made by building or importing this source locally.\n"""\nimport argparse\nimport json\nimport time\nimport traceback\nimport tempfile\nimport shutil\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom safetensors.torch import save_file\n\nfrom p3_training_utils import enable_training,forward_logits,wanted_weights,parameter_sample,synchronize,sha\nfrom p3_train_full import accumulation_groups,optimizer_update\n\n\ndef batches(indices,lengths,seed,training,microbatch=4,token_cap=1536):\n    """Keep every document and token; reduce batch size for long raw answers."""\n    indices=list(map(int,indices));order=sorted(indices,key=lambda i:(int(lengths[i]),i))\n    result=[];current=[];longest=0\n    for i in order:\n        length=int(lengths[i]);padded=((max(longest,length)+63)//64)*64\n        if current and (len(current)>=microbatch or padded*(len(current)+1)>token_cap):\n            result.append(current);current=[];longest=0\n        current.append(i);longest=max(longest,length)\n    if current:result.append(current)\n    if training:np.random.default_rng(seed).shuffle(result)\n    assert sorted(i for b in result for i in b)==sorted(indices)\n    return result\n\n\ndef run(a):\n    from qwen32_nf4 import Qwen32Classifier,EOS,REVISION\n    start=time.monotonic();out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\n    cfg=json.loads(Path(a.recipe).read_text())\n    assert cfg[\'backbone\']==\'Qwen3-32B\' and cfg[\'source_revision\']==REVISION\n    for key,path in [(\'train\',a.train),(\'external\',a.external)]:\n        assert sha(path)==cfg[key+\'_sha256\'],key\n    train=pd.read_csv(a.train,dtype=str).fillna(\'\')\n    external=pd.read_csv(a.external,dtype=str).fillna(\'\')\n    query=pd.read_csv(a.query,dtype=str).fillna(\'\')\n    assert len(train)==2699 and len(external)==cfg[\'external_rows\']\n    assert (external.text.str.len()>0).all() and external.model.isin(cfg[\'labels\']).all()\n    assert len(query.id)==query.id.nunique()\n    assert cfg[\'new_data_quality_selection\'] is False\n    assert cfg[\'official_epochs\']==2 and cfg[\'external_epochs\']==1\n    model=Qwen32Classifier(a.base,cfg[\'labels\'],seed=cfg[\'seed\'])\n    params=enable_training(model);assert sum(p.numel() for p in params.values())==17596434\n    frozen=parameter_sample(model)\n    tokens={name:model.tokenizer(frame.text.tolist(),add_special_tokens=False,truncation=False)[\'input_ids\']\n        for name,frame in [(\'external\',external),(\'official\',train),(\'query\',query)]}\n    for values in tokens.values():\n        assert max(map(len,values),default=0)+1<=model.model.config.max_position_embeddings\n    labels=model.labels\n    progress=dict(status=\'FRESH32_TRAIN_INFER_RUNNING\',source_revision=REVISION,\n        backbone=\'Qwen3-32B\',full64=True,uses27B=False,quality_measured=False,\n        external_rows=len(external),official_rows=len(train),query_rows=len(query),\n        text_truncation=False,trainable_parameters=17596434,completed=[],updates=0)\n\n    def write():\n        progress[\'elapsed_seconds\']=time.monotonic()-start\n        (out/\'progress.json\').write_text(json.dumps(progress,indent=2))\n\n    def before(stage):\n        now=time.time();remaining=a.deadline_wall-now\n        if remaining<120:raise TimeoutError(\'Global9h deadline reserve; no dummy predictions\')\n        if stage==\'training\' and time.monotonic()-start>a.train_max_seconds:\n            raise TimeoutError(\'Fixed full training budget exceeded; checkpoint saved\')\n\n    def pack(t,rows):\n        return model.tokenizer.pad({\'input_ids\':[t[i]+[EOS] for i in rows]},\n            padding=True,pad_to_multiple_of=64,return_tensors=\'pt\').to(model.device)\n\n    def save(stem,stage,**more):\n        path=out/(stem+\'.safetensors\')\n        save_file({n:p.detach().cpu().contiguous().clone() for n,p in wanted_weights(model).items()},str(path))\n        metadata=dict(model.metadata,stage=stage,benchmark_only=False,\n            not_for_scoring=stage!=\'final\',own_synthetic_fit_rows=len(external),\n            official_fit_rows=2699 if stage==\'final\' else 0,\n            official_adaptation_completed=stage==\'final\',RAW_exported=False,\n            recipe_sha256=sha(a.recipe),**more)\n        path.with_suffix(\'.json\').write_text(json.dumps(metadata,indent=2))\n        return path\n\n    def cached_training():\n        from prefix32 import capture,logits,gradient_parity\n        model.model.train();model.norms.train();model.plain.train()\n        # GPU parity is part of this genuine train-and-predict execution, not\n        # a separate quota-charged benchmark or an optimizer update.\n        t=tokens[\'external\'];lengths=np.asarray([len(x)+1 for x in t])\n        order=np.argsort(lengths,kind=\'stable\')\n        checks=[]\n        for indices in [order[max(0,len(order)//2-2):len(order)//2+2],order[-4:]]:\n            rows=list(map(int,indices))\n            targets=torch.tensor([labels.index(external.model.iloc[i]) for i in rows],device=model.device)\n            checks.append(gradient_parity(model,params,pack(t,rows),targets,forward_logits))\n        (out/\'prefix_parity.json\').write_text(json.dumps(dict(status=\'PASS\',checks=checks,\n            no_optimizer_updates=True,guard_fixed_before_metrics=True),indent=2))\n        cache=Path(tempfile.mkdtemp(prefix=\'ntl32-prefix-\',dir=\'/tmp\'))\n        assert shutil.disk_usage(\'/tmp\').free>60*2**30,\'Insufficient room for unmodified prefix activations\'\n        try:\n            for name,frame,epochs,lo,hi in [\n                (\'external\',external,1,1e-5,1e-4),(\'official\',train,2,2e-5,2e-4)]:\n                t=tokens[name];lengths=np.asarray([len(x)+1 for x in t])\n                y=np.asarray([labels.index(x) for x in frame.model])\n                fixed=batches(range(len(t)),lengths,0,False)\n                paths=[];cached_rows=0;capture_start=time.monotonic()\n                for i,rows in enumerate(fixed):\n                    before(\'training\');item=capture(model,pack(t,rows));item[\'rows\']=rows\n                    path=cache/f\'{name}_{i:05d}.pt\';torch.save(item,path);paths.append(path)\n                    cached_rows+=len(rows);del item\n                    if (i+1)%100==0:\n                        progress.update(phase=\'cache_\'+name,cached_rows=cached_rows,\n                            cache_seconds=time.monotonic()-capture_start)\n                        write();print(\'MERGE26_PREFIX_PROGRESS\',json.dumps(progress),flush=True)\n                assert cached_rows==len(frame)\n                model.model.train();model.norms.train();model.plain.train()\n                opt=torch.optim.AdamW([\n                    dict(params=[p for n,p in params.items() if n.startswith(\'backbone.\')],lr=lo),\n                    dict(params=[p for n,p in params.items() if not n.startswith(\'backbone.\')],lr=hi)],\n                    betas=(.9,.999),eps=1e-8,weight_decay=0)\n                scaler=torch.amp.GradScaler(\'cuda\',init_scale=128,growth_interval=10000)\n                exposures=0\n                for epoch in range(epochs):\n                    chunk_order=list(range(len(paths)));np.random.default_rng(cfg[\'seed\']+epoch).shuffle(chunk_order)\n                    groups=list(accumulation_groups([fixed[i] for i in chunk_order],target=16))\n                    row_to_path={tuple(rows):path for rows,path in zip(fixed,paths)}\n                    for group_number,group in enumerate(groups):\n                        before(\'training\');synchronize();update_start=time.monotonic()\n                        count=sum(map(len,group));opt.zero_grad(set_to_none=True);loss_sum=0.\n                        for rows in group:\n                            item=torch.load(row_to_path[tuple(rows)],map_location=\'cpu\',weights_only=True)\n                            assert item[\'rows\']==rows\n                            scores=logits(model,item)\n                            loss=torch.nn.functional.cross_entropy(scores,\n                                torch.tensor(y[rows],device=model.device),reduction=\'sum\')/count\n                            assert torch.isfinite(loss)\n                            scaler.scale(loss).backward();loss_sum+=float(loss.detach())\n                            del item,scores,loss\n                        for n,p in params.items():assert p.grad is not None and torch.isfinite(p.grad).all(),n\n                        scaler.unscale_(opt)\n                        torch.nn.utils.clip_grad_norm_(list(params.values()),1.,error_if_nonfinite=True)\n                        scaler.step(opt);scaler.update();synchronize()\n                        exposures+=count;progress[\'updates\']+=1\n                        progress.update(phase=name,epoch=epoch+1,row_exposures=exposures,\n                            frozen_prefix_cache=True,finite_gradients=True,loss_is_not_quality=loss_sum,\n                            last_update_rows=count,last_update_seconds=time.monotonic()-update_start)\n                        if progress[\'updates\']%25==0:\n                            path=save(\'RESUME_ONLY\',\'resume\',phase=name,epoch=epoch,next_group=group_number+1)\n                            torch.save(dict(optimizer=opt.state_dict(),scaler=scaler.state_dict(),\n                                phase=name,epoch=epoch,next_group=group_number+1,\n                                adapter_sha256=sha(path),recipe_sha256=sha(a.recipe),\n                                cpu_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all()),out/\'RESUME_ONLY.pt\')\n                            write();print(\'MERGE26_32_PROGRESS\',json.dumps(progress),flush=True)\n                assert exposures==len(t)*epochs\n                progress[\'completed\'].append(name);write()\n                for p in params.values():p.grad=None\n                save(name,\'external\' if name==\'external\' else \'official\')\n                for path in paths:path.unlink()\n            return save(\'fresh32_final\',\'final\',external_epochs=1,official_epochs=2,\n                frozen_prefix_cache=True,cache_dtype_preserved=True)\n        finally:\n            shutil.rmtree(cache)\n\n    try:\n        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction=False\n        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction=False\n        if a.trained_adapter:\n            metadata=json.loads(Path(a.trained_adapter).with_suffix(\'.json\').read_text())\n            assert metadata[\'stage\']==\'final\' and metadata[\'recipe_sha256\']==sha(a.recipe)\n            assert metadata[\'official_fit_rows\']==2699 and metadata[\'benchmark_only\'] is False\n            model.reload(a.trained_adapter)\n            progress[\'completed\'].append(\'matching_final_adapter_loaded\')\n            final=Path(a.trained_adapter)\n        elif cfg.get(\'frozen_prefix_cache\'):\n            final=cached_training()\n        else:\n            for name,frame,epochs,lo,hi in [\n                (\'external\',external,1,1e-5,1e-4),(\'official\',train,2,2e-5,2e-4)]:\n                t=tokens[name];lengths=np.asarray([len(x)+1 for x in t])\n                y=np.asarray([labels.index(x) for x in frame.model])\n                model.model.train();model.norms.train();model.plain.train()\n                opt=torch.optim.AdamW([\n                    dict(params=[p for n,p in params.items() if n.startswith(\'backbone.\')],lr=lo),\n                    dict(params=[p for n,p in params.items() if not n.startswith(\'backbone.\')],lr=hi)],\n                    betas=(.9,.999),eps=1e-8,weight_decay=0)\n                scaler=torch.amp.GradScaler(\'cuda\',init_scale=128,growth_interval=10000)\n                exposures=0\n                for epoch in range(epochs):\n                    groups=list(accumulation_groups(batches(range(len(t)),lengths,\n                        cfg[\'seed\']+epoch,True),target=16))\n                    for group_number,group in enumerate(groups):\n                        before(\'training\');synchronize();update_start=time.monotonic()\n                        count,loss,norm=optimizer_update(model,params,opt,scaler,group,t,y,pack)\n                        synchronize();exposures+=count;progress[\'updates\']+=1\n                        progress.update(phase=name,epoch=epoch+1,row_exposures=exposures,\n                            finite_gradients=True,loss_is_not_quality=loss,\n                            last_update_rows=count,last_update_seconds=time.monotonic()-update_start)\n                        if progress[\'updates\']%25==0:\n                            path=save(\'RESUME_ONLY\',\'resume\',phase=name,epoch=epoch,next_group=group_number+1)\n                            torch.save(dict(optimizer=opt.state_dict(),scaler=scaler.state_dict(),\n                                phase=name,epoch=epoch,next_group=group_number+1,\n                                adapter_sha256=sha(path),recipe_sha256=sha(a.recipe),\n                                cpu_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all()),out/\'RESUME_ONLY.pt\')\n                            write();print(\'MERGE26_32_PROGRESS\',json.dumps(progress),flush=True)\n                assert exposures==len(t)*epochs\n                progress[\'completed\'].append(name);write()\n                for p in params.values():p.grad=None\n                save(name,\'external\' if name==\'external\' else \'official\')\n            final=save(\'fresh32_final\',\'final\',external_epochs=1,official_epochs=2)\n        model.model.eval();model.norms.eval();model.plain.eval()\n        t=tokens[\'query\'];lengths=np.asarray([len(x)+1 for x in t]);scores=np.zeros((len(t),18),np.float32)\n        with torch.inference_mode():\n            for rows in batches(range(len(t)),lengths,0,False,microbatch=8,token_cap=3072):\n                before(\'inference\');scores[rows]=forward_logits(model,pack(t,rows)).cpu().numpy()\n        assert np.isfinite(scores).all()\n        if len(t):\n            rows=list(range(min(3,len(t))))\n            with torch.no_grad():before_reload=forward_logits(model,pack(t,rows)).cpu().numpy()\n            model.reload(final)\n            with torch.no_grad():after_reload=forward_logits(model,pack(t,rows)).cpu().numpy()\n            reload_error=float(np.max(np.abs(before_reload-after_reload)))\n            assert reload_error<.002\n        else:reload_error=0.\n        np.savez_compressed(out/\'fresh32_logits.npz\',ids=query.id.to_numpy(dtype=str),\n            labels=np.asarray(labels),logits=scores)\n        assert all(torch.equal(v,parameter_sample(model)[n]) for n,v in frozen.items())\n        progress.update(status=\'FRESH32_TRAIN_INFER_COMPLETE\',reload_error=reload_error,\n            final_adapter_sha256=sha(final),frozen_backbone_unchanged=True,\n            benchmark_weights_used=False,query_predictions_complete=True)\n        write();print(\'MERGE26_32_COMPLETE\',json.dumps(progress),flush=True)\n    except BaseException:\n        progress.update(status=\'FAILED\',traceback=traceback.format_exc());write();raise\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in [\'base\',\'external\',\'train\',\'query\',\'recipe\',\'output\']:p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--deadline-wall\',type=float,required=True)\n    p.add_argument(\'--train-max-seconds\',type=float,default=5*3600)\n    p.add_argument(\'--trained-adapter\')\n    run(p.parse_args())\n')
    import sys
    sys.path.insert(0,str(root))
    from pipeline import run
    run({'data': '/kaggle/input/competitions/name-that-llm', 'asset': '/kaggle/input/datasets/kirilltym/ntl-private-unfiltered-2026-10-26', 'labels': ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'], 'model_files': {'README.md': 15910, 'added_tokens.json': 707, 'chat_template.jinja': 4673, 'config.json': 1330, 'generation_config.json': 237, 'merges.txt': 1671853, 'model-00001-of-00004.safetensors': 4942503285, 'model-00002-of-00004.safetensors': 4963718184, 'model-00003-of-00004.safetensors': 4982643947, 'model-00004-of-00004.safetensors': 4323070149, 'model.safetensors.index.json': 275324, 'special_tokens_map.json': 614, 'tokenizer.json': 11422654, 'tokenizer_config.json': 10534, 'vocab.json': 2776833, 'LICENSE': 11343, 'PINNED_SOURCE.json': 5103}, 'mandatory_nlm_reserve_seconds': 10800, 'wheels': ['accelerate-1.15.0-py3-none-any.whl', 'bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', 'huggingface_hub-1.32.0-py3-none-any.whl', 'safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', 'tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', 'transformers-5.14.1-py3-none-any.whl']})


## Credits and licenses

The spectral readout method and public trained bank originate with **Kirill Tymchishin**, from [Teach More, Keep the Signature v1](https://www.kaggle.com/code/kirilltym/0-639-teach-more-keep-the-signature?scriptVersionId=361634630) and [NTL Spectral Style Heads — Paired 4404](https://www.kaggle.com/datasets/kirilltym/ntl-spectral-style-heads-paired-4404). The eight head formulas are retained with blocked-distance computation and a separate CPU worker. The original dataset metadata reports an unspecified license; this notebook references the original asset and does not relicense or redistribute its fitted banks.

The reference is [fleongg's NTL Part Two Complement, scored version 1](https://www.kaggle.com/code/fleongg/ntl-part-two-complement?scriptVersionId=361351359). It builds on [Nishant Dahal's two-stage public-pool LoRA](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool), [prvsiyan's shared-prompt Gaussian work](https://www.kaggle.com/code/prvsiyan/the-prompt-is-not-the-author), [Kirill Tymchishin's two-branch integration](https://www.kaggle.com/code/kirilltym/0-600-teach-the-model-remove-the-prompt), and [Travis Nguyen's in-style ensemble](https://www.kaggle.com/code/vnton37nguyn/ntl-lora-instyle-0-563).

Further component credit belongs to Lucas Boesen for LoRA ensembles and token-attention heads; Zbigniew Kirchner for Claude experts and in-style checkpoints; Yang Kuang Ou for n-gram features; Obstacle Developer for adaptive centered stylegram; pjmathematician for UltraBERT-380M; Joseph Adamski for the ATLAS implementation and early-layer work; and [mrpicklerick](https://www.kaggle.com/mrpicklerick) for the reviewed decoder-reference source. The ATLAS method is by Ko et al. Qwen, Meta, Microsoft and the other original model authors retain credit for their respective backbones. Public-pool contributions include Seyit Kaan Güneş, Abdullah Afify, Do Phuc Hung and the contributors cited by the linked references.

Original copyright notices remain in the visible custom modules. Attached code, datasets, models and adapters retain their individual licenses and terms; this notebook does not relicense them.
